# WP3B-M5 — Cached adapter replication across training seeds

Prepare exactly four fresh adapter arms: training seeds **43 and44**, each unbounded and relative-cap0.20. Split seed remains **42**. All ML implementation is visible below; no project processing modules are imported. The accepted M4 architecture, objective,323-candidate cosine scorer and checkpoint-selection policy are retained.

**Status: prepared, not executed.** Historical seed42 is a read-only reference: control selected epoch1 H@1=14.65%; bounded selected epoch3 H@1=19.04%, with a failed selected-checkpoint disease-safety gate (42/1496 unseen-diseased hits). These historical observations are not new replication results. No winner or significance claim is made for unexecuted seeds.

Use the existing WSL `as04_yolo_katana` kernel. Default Run All is metadata-only. Owner execution uses the verified feature cache; it never loads a CLIP encoder, downloads weights or encodes images.

In [ ]:
from pathlib import Path
from dataclasses import dataclass
from collections import Counter
from numbers import Integral, Real
from typing import Any
from datetime import datetime, timezone
import ast
import hashlib
import importlib.metadata as metadata
import json
import math
import platform
import random
import sys
import time
import unicodedata
import uuid
import numpy as np
import pandas as pd
import torch
import torchvision
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for directory in (start.resolve(), *start.resolve().parents):
        if (directory / "DPD_Small_train.csv").is_file() and (directory / "notebooks").is_dir():
            return directory
    raise FileNotFoundError("Open this notebook inside the WSL repository containing DPD_Small_train.csv.")
ROOT = find_project_root()
print({"python": platform.python_version(), "torch": str(torch.__version__),
       "torchvision": str(torchvision.__version__), "cuda_available": torch.cuda.is_available(),
       "repository": str(ROOT)})


## 2. Fixed configuration and separate seed identities

`split_seed` controls only the frozen partition; `training_seed` controls fresh adapter initialization and fitting-loader order. The proposed seeds, cap, losses, batch and five-epoch budgets are fixed; this is replication, not a hyperparameter sweep. Leave all three execution flags False until an explicit owner run.

In [ ]:
LOAD_FEATURE_CACHE = False
RUN_REPLICATIONS = False
BUILD_REPLICATION_REPORT = False
REPLICATION_PARENT = 'df2c9dd22657456780162b065d721b18e17a3810'
WP3A_PARENT = '300f1d78bb962f67fa2dbdb7b76ed23301abd7df'
CLIP_REVISION = 'd05afc436d78f1c48dc0dbf8e5980a9d471f35f6'
MODEL_REGISTRY = {'ViT-L/14': {'filename': 'ViT-L-14.pt',
              'sha256': 'b8cca3fd41ae0c99ba7e8951adf17d267cdb84cd88be6f7c2e0eca1737a03836',
              'embedding_dimension': 768,
              'input_resolution': 224}}
PROMPT_POLICY = {'version': 'label_only_single_template_v2',
 'normalization': 'Unicode NFKC; underscores to spaces; collapse whitespace; lowercase; no aliases/symptoms',
 'healthy': 'a photo of a healthy {crop} plant',
 'diseased': 'a photo of {article} {crop} plant affected by {disease}',
 'article_rule': 'an for normalized crop labels starting with a/e/i/o/u; otherwise a; diseased prompts only; '
                 'verified for the current 55 crop labels'}
COLUMN_NAMES = ['image_name', 'crop_id', 'disease_id', 'composition_id', 'crop_name', 'disease_name', 'composition_name']
SPLIT_VERSION = 'pytorch-v1'
DEFAULT_CSVS = ('DPD_Small_train.csv',)
EXPECTED_IDENTITIES = {'source_sha256': '9f1b729ab08f120c9e8d3a5681aa1ccbdf1d9210f9aaa6143959363dcb106a7d',
 'candidate_sha256': '4cb0a24a96cf647f32311817fa7b082a46d3cf00c20ae61f4287e83cc898dfaf',
 'split_sha256': 'f0c7ba212449fd68f1d19c0f84cc91bf4cc1af29ae405128e5198bce89f25f85'}
EXPECTED_DIRECT_PROMPT_SHA256 = '753672c7964bcb20a1346145f21a72b40a19f55590a006b5783823f23c13f537'
ADAPTER_SPEC = {'id': 'bounded_residual_visual_768_64_v1',
 'feature_dim': 768,
 'bottleneck': 64,
 'residual_scale': 0.1,
 'activation': 'ReLU',
 'bias': True,
 'up_initialization': 'zeros',
 'parameters': 99136}
PROTOTYPE_SPEC = {'source': 'only the 258 fitting-composition text embeddings; 323 direct candidates retained for evaluation',
 'aggregation': 'unit float64 text vectors; equal-candidate mean per original component ID; unit float64 '
                'then unit float32',
 'image_features_or_frequency_weights_used': False,
 'trainable': False}
LOSS_SPEC = {'crop_weight': 0.5,
 'disease_weight': 0.5,
 'identity_weight': 1.0,
 'temperature': 0.07,
 'identity': 'mean sample squared L2 between unit adapted and unit frozen features'}
NATIVE_PREPROCESSING = {'resize_shorter_edge': 224,
 'interpolation': 'bicubic',
 'center_crop': 224,
 'convert': 'RGB',
 'mean': [0.48145466, 0.4578275, 0.40821073],
 'std': [0.26862954, 0.26130258, 0.27577711],
 'augmentation': None}
SELECTION_POLICY = {'primary': 'maximum development H@1',
 'secondary': 'minimum sample-weighted combined validation objective',
 'tertiary': 'earliest epoch',
 'full_precision': True,
 'epoch_zero_eligible': True}
CANONICAL_PROJECT_ROOT = '/home/user1/COS30082 Applied Machine Learning/00 Assigments/AS02 ML Project'
WORKSPACE_DIRECTORY = '/home/user1/COS30082 Applied Machine Learning/00 Assigments/AS02 ML Project/workspace'
HANDOFF_DIRECTORY = '/home/user1/COS30082 Applied Machine Learning/00 Assigments/AS02 ML Project/workspace/handoffs'
M3_SPEC = {'id': 'same_crop_fitting_contrast_v1',
 'temperature': 0.07,
 'conditional_weight': [1.0],
 'negative_pool': '258 fitting-composition direct text vectors; same true crop only',
 'singleton': 'zero loss; mean over all processed fitting samples',
 'validation_loss': 'unchanged M2 component+identity objective',
 'inference': 'unchanged 323 direct float64 cosine scores'}
RESIDUAL_CAP_SPEC = {'choices': [None, 0.2],
 'relative_norm_bound': 'norm(y-x)/norm(x) <= rho',
 'max_angle_degrees': 11.536959032815489,
 'control': 'unbounded M3 adapter',
 'loss': 'unchanged M3 conditional objective'}
CONFIG = {'model': 'ViT-L/14',
 'batch_size': 8,
 'text_batch_size': 32,
 'num_workers': 0,
 'device': 'cuda',
 'learning_rate': 0.0001,
 'weight_decay': 0.0001,
 'epochs': 5,
 'conditional_weight': 1.0,
 'feature_cache_path': '/home/user1/COS30082 Applied Machine Learning/00 Assigments/AS02 ML '
                       'Project/workspace/experiments/wp3b-m4/feature_cache/20261104T180329Z_69b6bc6626a7.pt',
 'feature_cache_sha256': '9dbb07ae03ba38d003f5e4c610498a9a8c5ecb57475ec2f04f8e8481deeaa38d',
 'weight_directory': '/home/user1/COS30082 Applied Machine Learning/00 Assigments/AS02 ML '
                     'Project/workspace/pretrained_weights/clip',
 'image_directory': '/home/user1/COS30082 Applied Machine Learning/00 Assigments/AS02 ML '
                    'Project/workspace/datasets/DPD_Small_dataset',
 'output_directory': '/home/user1/COS30082 Applied Machine Learning/00 Assigments/AS02 ML '
                     'Project/workspace/experiments/wp3b-m5-replication',
 'split_seed': 42,
 'training_seeds': [43, 44]}
FEATURE_CACHE = None
REPLICATION_RESULTS = None
REPLICATION_REPORT = None
REPLICATION_STARTED = False
REPORT_GROUP_DIRECTORY = None
HISTORICAL_REFERENCE = {'control': {'directory': '/home/user1/COS30082 Applied Machine Learning/00 Assigments/AS02 ML '
                          'Project/workspace/experiments/wp3b-m4/20261104T180329Z_f8e7d239df6a',
             'files': {'adapter_summary.json': '954bcf790794451b5d0b0851cd26cd5984adf49b578ea697a4d82feaca4ee189',
                       'declaration.json': 'bf0cdb2db1fd44b78cf128748aa219678c9dc07919a51c6f73311dd5970f40e4',
                       'adapter_epoch_predictions.csv': '3a87b09b4ef8bc9ff02861aead53d6a79faab6afd2db4cf82bffd23801ffcc46',
                       'adapter_selected_predictions.csv': '47dcff9fc5c5051a1f454231820d6f3fb6260b64eb278ab36cf7f506d0a6a260',
                       'adapter_history.csv': 'a3cc752a7ac8b33b391b3b60e3a2860c449d86345b8038eb4342c5929f09ec34',
                       'direct_prompts.csv': 'bb7126e15849412b27ed8c0271c1e38ce756acd97d311b3dcbc37fd08ccff908',
                       'adapter_epoch_001_4b716f81.pt': '42e8ed0f10e5bc9496d2522ac5eb83fda3841dfedb61d4a3ea19994a211093c8'}},
 'bounded': {'directory': '/home/user1/COS30082 Applied Machine Learning/00 Assigments/AS02 ML '
                          'Project/workspace/experiments/wp3b-m4/20261104T180412Z_d9cd6d67c3a6',
             'files': {'adapter_summary.json': '3dcb7780971bb9a83753836fdcc2c0a12d11f3c394a3415e4c8ae3dc65bdfc03',
                       'declaration.json': '118fc239a69456042dd14d9b0c9db34c9183e053af917ba7aafc197e310f13ab',
                       'adapter_epoch_predictions.csv': 'f51bd82d96d5cfe999fe4900453ad26deb8d031f809ebe6b314f6ac0568721f6',
                       'adapter_selected_predictions.csv': '62eff3d84819b0bb3f65b6c1962c75038eb3138906669ff7c2c280943d833def',
                       'adapter_history.csv': '85c3eb9a2ea5fda000e76b5ab617a76c68e4bab8074e2355ca9cad2d287a0e3e',
                       'direct_prompts.csv': 'bb7126e15849412b27ed8c0271c1e38ce756acd97d311b3dcbc37fd08ccff908',
                       'adapter_epoch_003_68cf3edf.pt': '697d13afd3de6e5581ff22d9cfb0797871c614c251c651a97facaf149ce34d63'}}}


## 3. Training-source metadata and frozen three-way split

Read only the original training-source CSV. The visible accepted algorithm retains 4868 fitting /1238 seen validation /1894 internally unseen validation, full component coverage and non-contiguous original global composition IDs. No official-test metadata or image loader is used.

In [ ]:
def load_metadata(path):
    if Path(path).name != "DPD_Small_train.csv" or Path(path).resolve().name != "DPD_Small_train.csv":
        raise ValueError("M2 only reads training-source metadata.")
    frame = pd.read_csv(path, header=None)
    if frame.shape[1] != len(COLUMN_NAMES) or frame.empty:
        raise ValueError(f"Expected non-empty seven-column metadata: {path}")
    frame.columns = COLUMN_NAMES
    if frame.isna().any().any():
        raise ValueError(f"Missing metadata values: {path}")
    for col in ("crop_id", "disease_id", "composition_id"):
        numeric = pd.to_numeric(frame[col], errors="raise")
        if ((numeric < 0) | (numeric % 1 != 0)).any():
            raise ValueError(f"Invalid {col}: {path}")
        frame[col] = numeric.astype("int64")
    for name in frame.image_name:
        if not isinstance(name, str) or name in (".", "..") or any(c in name for c in ("/", "\\", ":")):
            raise ValueError(f"Expected flat image filename, got {name!r}")
    return frame


def validate_metadata(train, seen, unseen):
    frames = [train, seen, unseen]
    for frame in frames:
        if frame.image_name.duplicated().any():
            raise ValueError("Duplicate image names within a split")
    for i in range(3):
        for j in range(i + 1, 3):
            if set(frames[i].image_name) & set(frames[j].image_name):
                raise ValueError("Image filename overlap across official splits")
    all_rows = pd.concat(frames)
    for kind in ("crop", "disease", "composition"):
        pairs = all_rows[[f"{kind}_id", f"{kind}_name"]].drop_duplicates()
        if pairs[f"{kind}_id"].duplicated().any() or pairs[f"{kind}_name"].duplicated().any():
            raise ValueError(f"Inconsistent {kind} ID/name mapping")
    pairs = all_rows[["composition_id", "crop_id", "disease_id"]].drop_duplicates()
    if pairs.composition_id.duplicated().any() or pairs.duplicated(["crop_id", "disease_id"]).any():
        raise ValueError("Inconsistent composition/component mapping")
    if set(train.composition_id) & set(unseen.composition_id):
        raise ValueError("Official unseen compositions appear in training")
    if not set(seen.composition_id) <= set(train.composition_id):
        raise ValueError("Seen-test contains compositions absent from training")
    for col in ("crop_id", "disease_id"):
        if set(train[col]) != set(range(train[col].nunique())):
            raise ValueError(f"Training {col} must be contiguous and start at zero")
        if not set(all_rows[col]) <= set(train[col]):
            raise ValueError(f"Test {col} absent from training")
    return {"split_sizes": [len(f) for f in frames],
            "num_crops": int(train.crop_id.nunique()),
            "num_diseases": int(train.disease_id.nunique()),
            "num_seen_compositions": int(train.composition_id.nunique()),
            "num_unseen_compositions": int(unseen.composition_id.nunique()),
            "singleton_compositions": int((train.composition_id.value_counts() == 1).sum()),
            "filename_overlap": False, "seen_unseen_composition_overlap": False}


def make_validation_split(frame, val_split=0.2, seed=42, mode="stratified"):
    """Classwise split keeps singletons in train; heldout removes whole classes.

    Classwise rounding makes the realized image fraction approximate.
    heldout mode interprets val_split as a fraction of composition classes.
    """
    if not 0 < val_split < 1:
        raise ValueError("val_split must be between 0 and 1")
    frame = frame.reset_index(drop=True)
    rng = random.Random(seed)
    groups = {int(key): list(group.index) for key, group in frame.groupby("composition_id", sort=True)}
    val_indices = []
    if mode == "stratified":
        for indices in groups.values():
            rng.shuffle(indices)
            n = min(len(indices) - 1, max(1, round(len(indices) * val_split))) if len(indices) > 1 else 0
            val_indices.extend(indices[:n])
    elif mode == "heldout_composition":
        candidates = list(groups)
        rng.shuffle(candidates)
        remaining_crop = Counter(frame.crop_id)
        remaining_disease = Counter(frame.disease_id)
        target = max(1, round(len(groups) * val_split))
        heldout = []
        for comp_id in candidates:
            group = frame.loc[groups[comp_id]]
            crops, diseases = Counter(group.crop_id), Counter(group.disease_id)
            if all(remaining_crop[k] > n for k, n in crops.items()) and all(remaining_disease[k] > n for k, n in diseases.items()):
                heldout.append(comp_id)
                val_indices.extend(groups[comp_id])
                remaining_crop.subtract(crops)
                remaining_disease.subtract(diseases)
                if len(heldout) == target:
                    break
        if len(heldout) < target:
            raise ValueError(f"Only {len(heldout)} valid heldout classes; requested {target}. Reduce val_split.")
    else:
        raise ValueError(f"Unknown split mode: {mode}")
    validation = frame.loc[sorted(val_indices)].reset_index(drop=True)
    training = frame.drop(index=val_indices).reset_index(drop=True)
    if training.empty or validation.empty:
        raise ValueError("Split produced an empty training or validation set")
    for col in ("crop_id", "disease_id"):
        if set(frame[col]) != set(training[col]):
            raise ValueError(f"Split removed a training component: {col}")
    return training, validation


@dataclass(frozen=True)
class DevelopmentSplit:
    """Original labels and one candidate universe for both validation folds.

    Treat the contained DataFrames/mappings as read-only. Frozen field bindings
    do not make pandas objects immutable; DPDataset copies each frame.
    """

    fit: pd.DataFrame
    val_seen: pd.DataFrame
    val_unseen: pd.DataFrame
    mappings: dict[str, Any]
    candidate_ids: tuple[int, ...]
    candidate_pairs: tuple[tuple[int, int], ...]
    source_csv: str
    source_sha256: str
    candidate_sha256: str
    seed: int
    heldout_fraction: float
    seen_fraction: float
    split_version: str = f"{SPLIT_VERSION}/three-way-v1"


def _fraction(value: float, name: str) -> float:
    if isinstance(value, bool) or not isinstance(value, Real) or not 0 < value < 1:
        raise ValueError(f"{name} must be a finite fraction between zero and one")
    return float(value)


def _training_mappings(frame: pd.DataFrame) -> dict[str, Any]:
    """Build only training-derived maps; the combined train/test wrapper is unsuitable."""
    mappings = {}
    for kind in ("crop", "disease"):
        pairs = frame[[f"{kind}_id", f"{kind}_name"]].drop_duplicates()
        pairs = pairs.sort_values(f"{kind}_id")
        id_to_name = {int(key): str(name) for key, name in pairs.itertuples(index=False, name=None)}
        mappings[f"{kind}_id_to_name"] = id_to_name
        mappings[f"{kind}_name_to_id"] = {name: key for key, name in id_to_name.items()}
        mappings[f"num_{kind}s"] = len(id_to_name)
    pairs = frame[["composition_id", "crop_id", "disease_id", "composition_name"]]
    pairs = pairs.drop_duplicates().sort_values("composition_id")
    mappings["composition_id_to_pair"] = {
        int(row.composition_id): (int(row.crop_id), int(row.disease_id))
        for row in pairs.itertuples()
    }
    mappings["composition_id_to_name"] = {
        int(row.composition_id): str(row.composition_name) for row in pairs.itertuples()
    }
    ids = tuple(mappings["composition_id_to_pair"])
    mappings["composition_id_to_index"] = {comp: index for index, comp in enumerate(ids)}
    mappings["composition_index_to_id"] = dict(enumerate(ids))
    mappings["num_compositions"] = len(ids)
    return mappings


def _check_partitions(
    source: pd.DataFrame, fit: pd.DataFrame, seen: pd.DataFrame, unseen: pd.DataFrame,
) -> None:
    combined = pd.concat([fit, seen, unseen], ignore_index=True)
    if (combined.image_name.duplicated().any()
            or set(combined.image_name) != set(source.image_name)):
        raise ValueError("Development partitions must be disjoint and cover training filenames")
    if not source.sort_values("image_name").reset_index(drop=True).equals(
        combined.sort_values("image_name").reset_index(drop=True)
    ):
        raise ValueError("Development partitions changed original metadata rows or label types")
    fit_ids, seen_ids, unseen_ids = (set(frame.composition_id) for frame in (fit, seen, unseen))
    if not seen_ids <= fit_ids or unseen_ids & (fit_ids | seen_ids):
        raise ValueError("Seen coverage or held-out composition isolation failed")
    for column in ("crop_id", "disease_id"):
        if set(fit[column]) != set(source[column]):
            raise ValueError(f"Fitting partition lost training components: {column}")


def build_development_split(
    train_csv: str | Path = DEFAULT_CSVS[0],
    *,
    seed: int = 42,
    heldout_fraction: float = 0.2,
    seen_fraction: float = 0.2,
) -> DevelopmentSplit:
    """Compose existing train-only splits without reading images or writing files.

    The default DPD-Small source yields 4868 fit / 1238 seen / 1894 unseen
    images. heldout_fraction counts compositions; seen_fraction is classwise
    image sampling within the retained portion. Synthetic training CSVs are
    supported for tests. Known official test CSV paths are rejected.
    """
    if isinstance(seed, bool) or not isinstance(seed, Integral) or not 0 <= seed < 2**32:
        raise ValueError("seed must be an integer in [0, 2**32)")
    seed = int(seed)
    heldout_fraction = _fraction(heldout_fraction, "heldout_fraction")
    seen_fraction = _fraction(seen_fraction, "seen_fraction")
    path = Path(train_csv)
    if path.name != "DPD_Small_train.csv" or path.resolve().name != "DPD_Small_train.csv":
        raise ValueError("Official test metadata cannot be used as development training data")
    before_hash = hashlib.sha256(path.read_bytes()).hexdigest()
    source = load_metadata(path)
    if hashlib.sha256(path.read_bytes()).hexdigest() != before_hash:
        raise ValueError("Training CSV changed while loading")
    # Empty control frames reuse WP1 validation without loading either test CSV.
    empty = source.iloc[:0]
    validate_metadata(source, empty, empty)
    retained, unseen = make_validation_split(
        source, val_split=heldout_fraction, seed=seed, mode="heldout_composition"
    )
    fit, seen = make_validation_split(
        retained, val_split=seen_fraction, seed=seed, mode="stratified"
    )
    _check_partitions(source, fit, seen, unseen)
    mappings = _training_mappings(source)
    ids = tuple(mappings["composition_id_to_pair"])
    pairs = tuple(mappings["composition_id_to_pair"][comp] for comp in ids)
    records = [[comp, *pair] for comp, pair in zip(ids, pairs)]
    candidate_hash = hashlib.sha256(
        json.dumps(records, separators=(",", ":")).encode("utf-8")
    ).hexdigest()
    return DevelopmentSplit(
        fit=fit, val_seen=seen, val_unseen=unseen, mappings=mappings,
        candidate_ids=ids, candidate_pairs=pairs, source_csv=str(path.resolve()),
        source_sha256=before_hash, candidate_sha256=candidate_hash, seed=seed,
        heldout_fraction=heldout_fraction, seen_fraction=seen_fraction,
    )


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(4 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def json_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False).encode()).hexdigest()


def split_identity(split):
    records = {
        name: [[str(row.image_name), int(row.crop_id), int(row.disease_id), int(row.composition_id)]
               for row in frame.itertuples()]
        for name, frame in (("fit", split.fit), ("val_seen", split.val_seen), ("val_unseen", split.val_unseen))
    }
    return {"source_sha256": split.source_sha256, "candidate_sha256": split.candidate_sha256,
            "partition_sha256": {name: json_hash(rows) for name, rows in records.items()},
            "split_sha256": json_hash({name: frame.image_name.tolist() for name, frame in (("fit", split.fit), ("val_seen", split.val_seen), ("val_unseen", split.val_unseen))}), "split_version": split.split_version, "split_seed": split.seed}


## 4. Frozen prompts, native-CLIP provenance and direct composition scoring

Keep prompt policy `label_only_single_template_v2`. Cached embeddings were produced using native CLIP224-pixel bicubic resize/center-crop/RGB/CLIP normalization. Normalize embeddings and rank323 cosine scores in float64; exact ties use ascending global IDs. No WP2 head-score formula is substituted.

In [ ]:
def normalize_label(label):
    if not isinstance(label, str):
        raise TypeError("Labels must be strings.")
    cleaned = " ".join(unicodedata.normalize("NFKC", label).replace("_", " ").split()).lower()
    if not cleaned:
        raise ValueError("An empty label cannot form a prompt.")
    return cleaned


def crop_indefinite_article(crop_label):
    """Label-only rule, checked against the current 55-crop vocabulary."""
    crop = normalize_label(crop_label)
    return "an" if crop[0] in "aeiou" else "a"


def build_prompt_table(split):
    rows = []
    for column, (global_id, pair) in enumerate(zip(split.candidate_ids, split.candidate_pairs)):
        crop_id, disease_id = pair
        crop = normalize_label(split.mappings["crop_id_to_name"][crop_id])
        disease = normalize_label(split.mappings["disease_id_to_name"][disease_id])
        prompt = (PROMPT_POLICY["healthy"].format(crop=crop) if disease == "healthy"
                  else PROMPT_POLICY["diseased"].format(article=crop_indefinite_article(crop), crop=crop, disease=disease))
        rows.append({"score_column": column, "global_composition_id": global_id,
                     "crop_id": crop_id, "disease_id": disease_id, "crop": crop, "disease": disease,
                     "prompt": prompt})
    table = pd.DataFrame(rows)
    if len(table) != len(split.candidate_ids) or table.empty:
        raise ValueError("Candidate/prompt lengths disagree.")
    ids = table.global_composition_id.tolist()
    if ids != sorted(set(ids)) or list(zip(table.crop_id, table.disease_id)) != list(split.candidate_pairs):
        raise ValueError("Candidate IDs must be unique ascending global IDs with exact component alignment.")
    if table.prompt.duplicated().any():
        raise ValueError("Distinct candidates normalize to duplicate prompts; review the metadata.")
    return table


def unit_features(features):
    if not isinstance(features, torch.Tensor) or features.ndim != 2 or not features.is_floating_point():
        raise ValueError("Features must be a floating [N,D] tensor.")
    values = features.detach().to(device="cpu", dtype=torch.float64)
    if values.shape[0] == 0 or values.shape[1] == 0 or not torch.isfinite(values).all():
        raise ValueError("Feature vectors must be nonempty, finite and nonzero.")
    scale = values.abs().amax(dim=1, keepdim=True)
    if (scale == 0).any():
        raise ValueError("Zero feature vectors cannot be normalized.")
    scaled = values / scale
    return scaled / torch.linalg.vector_norm(scaled, dim=1, keepdim=True)


def cosine_candidate_scores(image_features, text_features):
    image = unit_features(image_features)
    text = unit_features(text_features)
    if image.shape[1] != text.shape[1]:
        raise ValueError("Image/text embedding dimensions differ.")
    scores = image @ text.T
    if not torch.isfinite(scores).all():
        raise ValueError("Non-finite cosine similarity.")
    return scores


def rank_global_ids(scores, candidate_ids):
    if not isinstance(scores, torch.Tensor) or scores.ndim != 2 or not scores.is_floating_point():
        raise ValueError("Scores must be a floating [N,C] tensor.")
    ids = tuple(candidate_ids)
    if not ids or any(type(i) is not int or i < 0 for i in ids) or list(ids) != sorted(set(ids)):
        raise ValueError("Candidates must be unique ascending nonnegative global IDs.")
    if scores.shape[1] != len(ids) or scores.shape[0] == 0 or not torch.isfinite(scores).all():
        raise ValueError("Score shape/values violate the candidate contract.")
    order = torch.argsort(scores.detach().to("cpu", torch.float64), dim=1, descending=True, stable=True)
    return torch.tensor(ids, dtype=torch.int64)[order]


def topk_accuracy(ranked_global_ids, truths, k):
    truths = torch.as_tensor(truths)
    if (ranked_global_ids.ndim != 2 or ranked_global_ids.dtype != torch.int64 or
            truths.ndim != 1 or truths.dtype != torch.int64 or len(truths) != len(ranked_global_ids)
            or len(truths) == 0):
        raise ValueError("Rankings/truths must be nonempty aligned int64 global IDs.")
    if type(k) is not int or not 1 <= k <= ranked_global_ids.shape[1]:
        raise ValueError("k must lie inside the candidate count.")
    if not (ranked_global_ids == truths.cpu()[:, None]).any(dim=1).all():
        raise ValueError("A composition truth is absent from the candidate universe.")
    correct = (ranked_global_ids[:, :k] == truths.cpu()[:, None]).any(dim=1)
    return correct.to(torch.float64).mean().item()


def harmonic_mean(seen, unseen):
    for value in (seen, unseen):
        if isinstance(value, bool) or not isinstance(value, Real) or not math.isfinite(value) or not 0 <= value <= 1:
            raise ValueError("Accuracies must be finite fractions in [0,1].")
    return 0.0 if seen + unseen == 0 else 2.0 * seen * unseen / (seen + unseen)


def fold_metrics(scores, truths, candidate_ids):
    ranked = rank_global_ids(scores, candidate_ids)
    return {"sample_count": len(ranked), "top1": topk_accuracy(ranked, truths, 1),
            "top5": topk_accuracy(ranked, truths, 5)}, ranked


def weight_spec(model_name):
    spec = dict(MODEL_REGISTRY[model_name])
    spec["url"] = f"https://openaipublic.azureedge.net/clip/models/{spec['sha256']}/{spec['filename']}"
    return spec


## 5. Identity-initialized adapter and fixed component objective

The trainable768→64→768 residual has99136 parameters; only its parameters enter AdamW. The same training seed initializes both cap choices identically. Raw frozen features are detached; crop/disease anchors use only258 fitting-composition texts. Non-finite residuals/norms/outputs fail explicitly.

In [ ]:
class ResidualVisualAdapter(torch.nn.Module):
    """A 99,136-parameter visual residual; exact raw-feature identity at initialization."""
    def __init__(self, residual_cap=None):
        super().__init__()
        if residual_cap is not None and (type(residual_cap) not in (int, float) or residual_cap != .2):
            raise ValueError("Only the predeclared unbounded control or rho=0.20 is supported.")
        self.residual_cap = residual_cap
        self.down = torch.nn.Linear(768, 64)
        self.up = torch.nn.Linear(64, 768)
        torch.nn.init.kaiming_uniform_(self.down.weight, a=math.sqrt(5))
        torch.nn.init.zeros_(self.down.bias)
        torch.nn.init.zeros_(self.up.weight)
        torch.nn.init.zeros_(self.up.bias)

    def forward(self, features):
        if not isinstance(features, torch.Tensor) or features.ndim != 2 or features.shape[1] != 768:
            raise ValueError("Expected frozen CLIP image embeddings [B,768].")
        base = features.detach().float()
        norms = torch.linalg.vector_norm(base, dim=1, keepdim=True)
        if (not len(base) or not torch.isfinite(base).all()
                or not torch.isfinite(norms).all() or (norms <= 0).any()):
            raise ValueError("Image features must be finite and nonzero.")
        unit = base / norms
        residual = self.up(torch.relu(self.down(unit)))
        if not torch.isfinite(residual).all():
            raise ValueError("Non-finite learned adapter residual.")
        # Preserve the accepted arithmetic for every finite valid input in both arms.
        if self.residual_cap is None:
            output = base + 0.1 * norms * residual  # Exact M3 control arithmetic.
        else:
            delta = 0.1 * residual
            size = torch.linalg.vector_norm(delta, dim=1, keepdim=True)
            if not torch.isfinite(size).all():
                raise ValueError("Non-finite relative residual size; no norm guarantee is possible.")
            factor = (self.residual_cap / size.clamp_min(torch.finfo(delta.dtype).eps)).clamp(max=1.0)
            output = base + norms * (delta * factor)
        output_norms = torch.linalg.vector_norm(output, dim=1)
        if (not torch.isfinite(output).all() or not torch.isfinite(output_norms).all()
                or (output_norms <= 0).any()):
            raise ValueError("Adapter output must be finite and nonzero.")
        return output


def require_frozen_clip(model):
    if model.training or any(p.requires_grad or p.grad is not None for p in model.parameters()):
        raise ValueError("CLIP must be eval-mode, frozen, and have no accumulated gradients.")


@torch.no_grad()
def component_prototypes(direct_features, table, fitting_candidate_ids=None):
    vectors = unit_features(direct_features)
    if len(vectors) != len(table):
        raise ValueError("Direct text rows and candidate table differ.")
    if fitting_candidate_ids is None:
        raise ValueError("Explicit fitting-only candidate IDs are required.")
    pool = list(fitting_candidate_ids)
    if not pool or pool != sorted(set(pool)) or not set(pool) <= set(table.global_composition_id):
        raise ValueError("Invalid fitting-only prototype pool.")
    mask = table.global_composition_id.isin(pool).to_numpy()
    vectors = vectors[mask]
    table = table.loc[mask].reset_index(drop=True)
    result = {}
    for kind in ("crop", "disease"):
        ids = tuple(int(x) for x in sorted(table[kind + "_id"].unique()))
        means = torch.stack([vectors[table[kind + "_id"].to_numpy() == key].mean(0) for key in ids])
        # Fixed fitting-composition text averages; no image features or frequency weights.
        anchors = unit_features(means).float().clone()
        result[kind] = torch.nn.functional.normalize(anchors, dim=1).detach()
        result[kind + "_ids"] = ids
    return result


def adapter_objective(adapted, original, crop_labels, disease_labels, prototypes):
    if (adapted.ndim != 2 or adapted.shape != original.shape or adapted.shape[1] != 768
            or not len(adapted) or not torch.isfinite(adapted).all()
            or not torch.isfinite(original).all()
            or (torch.linalg.vector_norm(original.detach().float(), dim=1) <= 0).any()
            or (torch.linalg.vector_norm(adapted.float(), dim=1) <= 0).any()):
        raise ValueError("Loss inputs must be finite aligned [B,768] embeddings.")
    y = torch.nn.functional.normalize(adapted.float(), dim=1)
    x = torch.nn.functional.normalize(original.detach().float(), dim=1)
    losses = {}
    for kind, labels in (("crop", crop_labels), ("disease", disease_labels)):
        anchor = prototypes[kind]
        ids = tuple(prototypes[kind + "_ids"])
        if (anchor.ndim != 2 or anchor.shape != (len(ids), 768) or anchor.requires_grad
                or not torch.isfinite(anchor).all() or len(set(ids)) != len(ids)):
            raise ValueError("Component anchors must be fixed, finite and correctly mapped.")
        if (not isinstance(labels, torch.Tensor) or labels.dtype != torch.int64
                or labels.shape != (len(y),)):
            raise ValueError("Labels must be aligned original int64 component IDs.")
        columns = {key: col for col, key in enumerate(ids)}
        values = labels.detach().cpu().tolist()
        if not set(values) <= set(columns):
            raise ValueError("Component label is absent from the fixed prototype map.")
        target = torch.tensor([columns[v] for v in values], dtype=torch.int64, device=y.device)
        logits = y @ anchor.to(device=y.device, dtype=torch.float32).T / 0.07
        losses[kind + "_loss"] = torch.nn.functional.cross_entropy(logits, target)
    identity = ((y - x) ** 2).sum(dim=1).mean()
    total = .5 * losses["crop_loss"] + .5 * losses["disease_loss"] + 1.0 * identity
    if not torch.isfinite(total):
        raise ValueError("Non-finite adapter objective.")
    return {"loss": total, **losses, "identity_loss": identity}


def make_adapter_optimizer(adapter, config):
    params = list(adapter.parameters())
    if len(set(id(p) for p in params)) != len(params) or not all(p.requires_grad for p in params):
        raise ValueError("Expected distinct trainable adapter parameters only.")
    return torch.optim.AdamW(params, lr=config["learning_rate"], weight_decay=config["weight_decay"])


def tensor_sha256(tensor):
    return hashlib.sha256(tensor.detach().cpu().contiguous().numpy().tobytes()).hexdigest()


def seal_text_state(direct, prototypes, contract):
    # Clone outside inference_mode so fixed anchors can be saved for adapter backward.
    state = {"direct": direct.detach().cpu().clone(),
             "crop": prototypes["crop"].detach().cpu().float().clone(),
             "disease": prototypes["disease"].detach().cpu().float().clone(),
             "crop_ids": list(prototypes["crop_ids"]), "disease_ids": list(prototypes["disease_ids"]),
             "candidate_ids": list(contract["candidate_ids"]),
             "prototype_pool_ids": list(contract["fitting_composition_ids"]), "contract_sha256": json_hash(contract)}
    state["hashes"] = {key: tensor_sha256(state[key]) for key in ("direct", "crop", "disease")}
    state.update(build_conditional_state(state["direct"], contract))
    validate_text_state(state, contract)
    return state


def validate_text_state(state, contract):
    if state["contract_sha256"] != json_hash(contract):
        raise ValueError("Text state belongs to a different predeclared experiment.")
    if state["candidate_ids"] != contract["candidate_ids"]:
        raise ValueError("Reordered global candidate mapping.")
    if state["prototype_pool_ids"] != contract["fitting_composition_ids"]:
        raise ValueError("Prototype training-pool provenance changed.")
    for kind, count in (("direct", 323), ("crop", 55), ("disease", 175)):
        tensor = state[kind]
        if (tensor.shape != (count, 768) or tensor.requires_grad
                or tensor.dtype != (torch.float64 if kind == "direct" else torch.float32)
                or not torch.isfinite(tensor).all()
                or tensor_sha256(tensor) != state["hashes"][kind]):
            raise ValueError("Text/prototype bank changed: " + kind)
    for kind in ("crop", "disease"):
        if list(state[kind + "_ids"]) != contract["component_ids"][kind]:
            raise ValueError("Reordered component mapping.")
    validate_conditional_state(state, contract)
    return True


def check_batch_targets(targets, expected, seen_names):
    names = list(targets["image_name"])
    if not names or len(set(names)) != len(names) or set(names) & seen_names:
        raise ValueError("Empty or repeated image identities.")
    if not set(names) <= set(expected.index):
        raise ValueError("Batch contains a filename outside the approved partition.")
    rows = expected.loc[names]
    for key, column in (("crop", "crop_id"), ("disease", "disease_id"), ("composition", "composition_id")):
        labels = torch.as_tensor(targets[key])
        if labels.dtype != torch.int64 or labels.shape != (len(names),):
            raise ValueError("Original int64 label compatibility failed.")
        if labels.cpu().tolist() != rows[column].tolist():
            raise ValueError("Batch labels no longer match the source filename.")
    seen_names.update(names)
    return names


## 6. Unchanged same-crop fitting supervision

Retain M3 conditional weight1 and temperature0.07, plus the existing component and identity objective. Negative composition texts come only from the258 fitting compositions sharing the true crop. No held-out image or composition-label supervision enters optimization.

In [ ]:
def build_conditional_state(direct, contract):
    ids = list(contract["candidate_ids"])
    pool = list(contract["fitting_composition_ids"])
    if pool != sorted(set(pool)) or not pool or not set(pool) <= set(ids):
        raise ValueError("Explicit ordered fitting-only conditional pool required.")
    positions = [ids.index(i) for i in pool]
    pairs = [contract["candidate_pairs"][i] for i in positions]
    text = torch.nn.functional.normalize(direct[positions].detach().cpu().float().clone(), dim=1)
    return {"conditional_ids": torch.tensor(pool, dtype=torch.int64),
            "conditional_pairs": torch.tensor(pairs, dtype=torch.int64),
            "conditional_text": text, "conditional_sha256": tensor_sha256(text),
            "conditional_weight": float(contract["hyperparameters"]["conditional_weight"])}


def validate_conditional_state(state, contract):
    expected = build_conditional_state(state["direct"], contract)
    for key in ("conditional_ids", "conditional_pairs", "conditional_text"):
        actual = state[key]
        if (not isinstance(actual, torch.Tensor) or actual.requires_grad
                or actual.dtype != expected[key].dtype or not torch.equal(actual.cpu(), expected[key])):
            raise ValueError("Changed fitting-only conditional bank: " + key)
    if (state["conditional_sha256"] != tensor_sha256(state["conditional_text"])
            or state["conditional_weight"] != expected["conditional_weight"]):
        raise ValueError("Conditional objective/hash mismatch.")


def same_crop_loss(adapted, crop_labels, disease_labels, composition_labels, state):
    """CE among fitting descriptions of the true crop; never used for inference."""
    if (adapted.ndim != 2 or adapted.shape[1] != 768 or not len(adapted)
            or not torch.isfinite(adapted).all()
            or not torch.isfinite(torch.linalg.vector_norm(adapted.float(), dim=1)).all()
            or (torch.linalg.vector_norm(adapted.float(), dim=1) <= 0).any()):
        raise ValueError("Conditional loss expects finite nonzero [B,768] features.")
    labels = (crop_labels, disease_labels, composition_labels)
    if any(not isinstance(y, torch.Tensor) or y.dtype != torch.int64 or y.shape != (len(adapted),) for y in labels):
        raise ValueError("Aligned original int64 crop/disease/composition labels required.")
    ids, pairs, bank = (state[k] for k in ("conditional_ids", "conditional_pairs", "conditional_text"))
    if (ids.ndim != 1 or ids.dtype != torch.int64 or not len(ids)
            or pairs.shape != (len(ids), 2) or pairs.dtype != torch.int64
            or bank.shape != (len(ids), 768) or bank.requires_grad
            or not torch.isfinite(bank).all() or len(set(ids.tolist())) != len(ids)):
        raise ValueError("Malformed frozen conditional bank.")
    lookup = {int(key): j for j, key in enumerate(ids.tolist())}
    truth = composition_labels.detach().cpu().tolist()
    if not set(truth) <= set(lookup):
        raise ValueError("Held-out/unknown compositions cannot enter fitting supervision.")
    columns = torch.tensor([lookup[key] for key in truth], dtype=torch.int64)
    expected = pairs.cpu()[columns]
    if (not torch.equal(expected[:, 0], crop_labels.detach().cpu())
            or not torch.equal(expected[:, 1], disease_labels.detach().cpu())):
        raise ValueError("Composition/component labels disagree.")
    y = torch.nn.functional.normalize(adapted.float(), dim=1)
    logits = y @ bank.to(y.device).T / .07
    mask = pairs[:, 0].to(y.device)[None, :] == crop_labels.to(y.device)[:, None]
    if not mask.any(dim=1).all():
        raise ValueError("No fitting descriptions for a training crop.")
    # -inf excludes other crops exactly; CE/logsumexp stays finite because each row has its target.
    loss = torch.nn.functional.cross_entropy(logits.masked_fill(~mask, float("-inf")), columns.to(y.device))
    if not torch.isfinite(loss):
        raise ValueError("Non-finite conditional objective.")
    return loss


def m3_training_objective(adapted, original, crop_labels, disease_labels, composition_labels, state, weight):
    if type(weight) not in (int, float) or weight not in (0., 1.):
        raise ValueError("Only predeclared control/intervention weights 0 or 1 are supported.")
    base = adapter_objective(adapted, original, crop_labels, disease_labels, state)
    conditional = same_crop_loss(adapted, crop_labels, disease_labels, composition_labels, state)
    total = base["loss"] if weight == 0 else base["loss"] + conditional
    if not torch.isfinite(total):
        raise ValueError("Non-finite M3 training objective.")
    return {**base, "loss": total, "conditional_loss": conditional}


## 7. Visible fitting, validation and deterministic selection

Each fitting image is processed exactly once per epoch. Validation uses inference mode, full ordered folds and sample-weighted losses. Selection maximizes development H@1, then minimizes combined validation loss, then chooses the earliest epoch. Epoch0 remains eligible. Disease diagnostics do not retrospectively replace the selected epoch.

In [ ]:
def train_adapter_epoch(model, adapter, loader, optimizer, state, fit_frame, device, progress=True):
    from tqdm.auto import tqdm
    require_frozen_clip(model)
    if getattr(loader.dataset, "role", None) != "fit" or not len(loader) or fit_frame.empty:
        raise ValueError("Optimization requires a nonempty fitting-only loader.")
    owned = {id(p) for group in optimizer.param_groups for p in group["params"]}
    if owned != {id(p) for p in adapter.parameters()}:
        raise ValueError("Optimizer must contain adapter parameters only.")
    adapter.train()
    expected = fit_frame.set_index("image_name", drop=False)
    seen_names, count = set(), 0
    totals = {k: 0. for k in ("loss", "crop_loss", "disease_loss", "identity_loss", "conditional_loss")}
    device = torch.device(device)
    if device.type == "cuda":
        torch.cuda.synchronize(device)
        torch.cuda.reset_peak_memory_stats(device)
    started = time.perf_counter()
    for images, targets in tqdm(loader, desc="Fitting adapter only", disable=not progress):
        names = check_batch_targets(targets, expected, seen_names)
        if len(images) != len(names) or not torch.isfinite(images).all():
            raise ValueError("Invalid fitting image batch.")
        optimizer.zero_grad(set_to_none=True)
        # no_grad rather than inference_mode: these constants are saved for adapter backward.
        with torch.no_grad():
            original = model.encode_image(images.to(device)).detach().float()
        adapted = adapter(original)
        losses = m3_training_objective(adapted, original, targets["crop"], targets["disease"],
                                       targets["composition"], state, state["conditional_weight"])
        losses["loss"].backward()
        gradients = [p.grad for p in adapter.parameters()]
        if any(g is None or not torch.isfinite(g).all() for g in gradients):
            raise ValueError("Missing or non-finite adapter gradient.")
        torch.nn.utils.clip_grad_norm_(adapter.parameters(), 1.0, error_if_nonfinite=True)
        optimizer.step()
        require_frozen_clip(model)
        n = len(names)
        for key in totals:
            totals[key] += float(losses[key].detach()) * n
        count += n
    if seen_names != set(expected.index) or count != len(fit_frame):
        raise ValueError("Each fitting image must be processed exactly once.")
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    return {**{k: v / count for k, v in totals.items()}, "sample_count": count,
            "elapsed_seconds": time.perf_counter() - started,
            "peak_allocated_bytes": torch.cuda.max_memory_allocated(device) if device.type == "cuda" else None,
            "peak_reserved_bytes": torch.cuda.max_memory_reserved(device) if device.type == "cuda" else None}


@torch.inference_mode()
def validate_adapter(model, adapter, loader, state, expected_frame, candidate_ids, device, progress=True):
    from tqdm.auto import tqdm
    require_frozen_clip(model)
    if getattr(loader.dataset, "role", None) not in ("val_seen", "val_unseen") or not len(loader):
        raise ValueError("Only nonempty development validation loaders are accepted.")
    adapter.eval()
    if list(candidate_ids) != state["candidate_ids"]:
        raise ValueError("Validation candidate ordering mismatch.")
    device = torch.device(device)
    expected = expected_frame.set_index("image_name", drop=False)
    seen_names, filenames, truths, ranks, controls = set(), [], [], [], []
    totals = {k: 0. for k in ("loss", "crop_loss", "disease_loss", "identity_loss")}
    count = 0
    if device.type == "cuda":
        torch.cuda.synchronize(device)
        torch.cuda.reset_peak_memory_stats(device)
    started = time.perf_counter()
    for images, targets in tqdm(loader, desc=loader.dataset.role, disable=not progress):
        names = check_batch_targets(targets, expected, seen_names)
        if len(images) != len(names) or not torch.isfinite(images).all():
            raise ValueError("Invalid validation batch.")
        original = model.encode_image(images.to(device)).float()
        adapted = adapter(original)
        losses = adapter_objective(adapted, original, targets["crop"], targets["disease"], state)
        # Unchanged primary scorer: raw adapted features against the original 323 text vectors.
        scores = cosine_candidate_scores(adapted, state["direct"])
        control = cosine_candidate_scores(original, state["direct"])
        ranks.append(rank_global_ids(scores, candidate_ids))
        controls.append(rank_global_ids(control, candidate_ids))
        truths.append(targets["composition"].cpu())
        filenames.extend(names)
        n = len(names)
        for key in totals:
            totals[key] += float(losses[key]) * n
        count += n
    if filenames != expected_frame.image_name.tolist() or count != len(expected_frame):
        raise ValueError("Validation must preserve the full ordered partition exactly.")
    ranking, control_ranking, truth = torch.cat(ranks), torch.cat(controls), torch.cat(truths)
    metrics = {"top1": topk_accuracy(ranking, truth, 1), "top5": topk_accuracy(ranking, truth, 5)}
    control_metrics = {"top1": topk_accuracy(control_ranking, truth, 1),
                       "top5": topk_accuracy(control_ranking, truth, 5)}
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    return {**{k: v / count for k, v in totals.items()}, "sample_count": count,
        "metrics": metrics, "control_metrics": control_metrics, "ranking": ranking,
        "control_ranking": control_ranking, "truths": truth, "filenames": filenames,
        "elapsed_seconds": time.perf_counter() - started,
        "peak_allocated_bytes": torch.cuda.max_memory_allocated(device) if device.type == "cuda" else None,
        "peak_reserved_bytes": torch.cuda.max_memory_reserved(device) if device.type == "cuda" else None}


def selection_record(epoch, seen, unseen):
    if type(epoch) is not int or epoch < 0:
        raise ValueError("Selection requires a nonnegative epoch.")
    for result, expected in ((seen, 1238), (unseen, 1894)):
        count = result["sample_count"]
        if isinstance(count, bool) or not isinstance(count, Integral) or count != expected:
            raise ValueError("Selection requires the complete approved 1238 seen / 1894 unseen folds.")
    combined = (seen["sample_count"] * seen["loss"] + unseen["sample_count"] * unseen["loss"]) / (
        seen["sample_count"] + unseen["sample_count"])
    if not math.isfinite(combined):
        raise ValueError("Non-finite combined validation loss.")
    return {"epoch": epoch, "h1": harmonic_mean(seen["metrics"]["top1"], unseen["metrics"]["top1"]),
        "h5": harmonic_mean(seen["metrics"]["top5"], unseen["metrics"]["top5"]),
        "seen_top1": seen["metrics"]["top1"], "seen_top5": seen["metrics"]["top5"],
        "unseen_top1": unseen["metrics"]["top1"], "unseen_top5": unseen["metrics"]["top5"],
        "seen_count": seen["sample_count"], "unseen_count": unseen["sample_count"],
        "combined_validation_loss": combined, "selection_policy": dict(SELECTION_POLICY)}


def better_selection(candidate, best):
    def key(record):
        if (not math.isfinite(record["h1"]) or not 0 <= record["h1"] <= 1
                or not math.isfinite(record["combined_validation_loss"])
                or type(record["epoch"]) is not int or record["epoch"] < 0):
            raise ValueError("Invalid checkpoint-selection record.")
        return (-record["h1"], record["combined_validation_loss"], record["epoch"])
    candidate_key = key(candidate)
    return best is None or candidate_key < key(best)


## 8. Safe adapter checkpoints and code identity

Reuse the accepted M4 adapter state-dictionary payload and strict reconstruction without an encoder or download. Each new experiment has an explicit M5 contract with separate split/training seeds. File creation is exclusive; historical files are never overwritten.

In [ ]:
def require_external_path(path):
    """Compatibility name: outputs now belong to the ignored canonical workspace."""
    path = Path(path)
    if not path.is_absolute():
        raise ValueError("Use an absolute canonical workspace path.")
    project = Path(CANONICAL_PROJECT_ROOT).resolve()
    workspace = Path(WORKSPACE_DIRECTORY)
    if workspace != project / "workspace" or workspace.resolve() != workspace:
        raise ValueError("The canonical workspace must not redirect outside the project.")
    allowed = workspace / "experiments"
    if allowed.resolve() != allowed:
        raise ValueError("The experiments directory must not redirect outside the workspace.")
    path = path.resolve()
    if path != allowed and allowed not in path.parents:
        raise ValueError("M2 outputs must stay inside canonical workspace/experiments/.")
    for directory in (path, *path.parents):
        if (directory / ".git").exists():
            raise ValueError("Experiment outputs must not enter a nested Git checkout.")
        if directory == allowed:
            break  # The enclosing canonical repository is deliberately permitted.
    return path


def save_adapter_checkpoint(path, adapter, text_state, contract, record):
    validate_text_state(text_state, contract)
    if type(record.get("epoch")) is not int or record["epoch"] < 0:
        raise ValueError("Checkpoint metadata must contain a nonnegative epoch.")
    if not isinstance(adapter, ResidualVisualAdapter):
        raise ValueError("Only the declared residual adapter can be checkpointed.")
    path = require_external_path(path)
    json.dumps(contract, allow_nan=False)
    json.dumps(record, allow_nan=False)
    if any(not torch.isfinite(t).all() for t in adapter.state_dict().values()):
        raise ValueError("Non-finite adapter parameters cannot be saved.")
    if adapter.residual_cap != contract["hyperparameters"]["residual_cap"]:
        raise ValueError("Adapter cap disagrees with the declared experiment.")
    payload = {"schema": "wp3b-visual-adapter-m4-v1", "adapter_spec": dict(contract["adapter"]),
        "adapter_state": {k: v.detach().cpu().clone() for k, v in adapter.state_dict().items()},
        "contract": contract, "contract_sha256": json_hash(contract),
        "text_state": {k: (v.detach().cpu().clone() if isinstance(v, torch.Tensor) else v)
                       for k, v in text_state.items()},
        "selection": record}
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("xb") as handle:  # Never overwrite an existing checkpoint.
        torch.save(payload, handle)
    return str(path)


def load_adapter_checkpoint(path, expected_contract):
    payload = torch.load(Path(path), map_location="cpu", weights_only=True)
    if payload.get("schema") != "wp3b-visual-adapter-m4-v1" or payload.get("adapter_spec") != expected_contract["adapter"]:
        raise ValueError("Unsupported adapter schema or architecture.")
    if (payload["contract_sha256"] != json_hash(payload["contract"])
            or payload["contract_sha256"] != json_hash(expected_contract)
            or payload["contract"] != expected_contract):
        raise ValueError("Model/source/split/candidate/component/preprocessing contract mismatch.")
    validate_text_state(payload["text_state"], expected_contract)
    adapter = ResidualVisualAdapter(expected_contract["hyperparameters"]["residual_cap"])
    adapter.load_state_dict(payload["adapter_state"], strict=True)
    if any(not torch.isfinite(t).all() for t in adapter.state_dict().values()):
        raise ValueError("Non-finite adapter checkpoint.")
    adapter.eval()
    # This reconstructs the small adapter only; it never loads/downloads CLIP.
    return adapter, payload["text_state"], payload["selection"]


def verify_split_identity(split):
    identity = split_identity(split)
    if sha256_file(split.source_csv) != identity["source_sha256"]:
        raise ValueError("Training-source bytes changed.")
    if any(identity[k] != v for k, v in EXPECTED_IDENTITIES.items()):
        raise ValueError("Historical source/split/candidate identity changed.")
    if [len(split.fit), len(split.val_seen), len(split.val_unseen)] != [4868, 1238, 1894]:
        raise ValueError("Unexpected development partition sizes.")
    return identity


def notebook_algorithm_hash():
    book = json.loads((ROOT / "notebooks/wp3b_seed_replication.ipynb").read_text())
    nodes = [ast.dump(n, include_attributes=False) for c in book["cells"] if c["cell_type"] == "code"
             for n in ast.parse("".join(c["source"])).body if isinstance(n, (ast.FunctionDef, ast.ClassDef))]
    return json_hash(nodes)


## 9. Verified immutable feature cache and fitting-only DataLoaders

Load the explicitly pinned artifact with `weights_only=True`; verify tensor/label/filename hashes, original source/split/candidate/prompt identities, model/native transform provenance, current8000 approved image byte hashes and exact historical frozen Top-5 parity. Only raw-byte validation reads current image files; no image decoding or encoding is performed. Process-local function addresses are normalized without weakening transform settings or rewriting legacy hashes.

In [ ]:
def canonical_native_preprocess_repr(value):
    """Remove only CLIP's process-local RGB-converter address; retain all transform settings."""
    import re
    if not isinstance(value, str):
        raise ValueError("Native preprocessing provenance must be a string.")
    return re.sub(r"<function _convert_image_to_rgb at 0x[0-9a-fA-F]+>",
                  "<function _convert_image_to_rgb>", value)


def native_preprocess_repr():
    from clip.clip import _transform
    return canonical_native_preprocess_repr(repr(_transform(224)))


def expected_cache_spec(split, table, config):
    if config["model"] != "ViT-L/14" or not table.equals(build_prompt_table(split)):
        raise ValueError("Only the fixed native ViT-L/14 development cache is supported.")
    prompt_hash = json_hash({"policy":PROMPT_POLICY,"rows":table.to_dict("records")})
    if prompt_hash != EXPECTED_DIRECT_PROMPT_SHA256:
        raise ValueError("Prompt policy/order changed.")
    return {"version":"native-frozen-clip-raw-f32-v1", "split":verify_split_identity(split),
        "candidate_ids":list(split.candidate_ids),"candidate_pairs":[list(v) for v in split.candidate_pairs],
        "model":"ViT-L/14","model_sha256":MODEL_REGISTRY["ViT-L/14"]["sha256"],
        "CLIP_revision":CLIP_REVISION,"prompt_sha256":prompt_hash,
        "preprocessing":dict(NATIVE_PREPROCESSING),"native_transform":native_preprocess_repr(),
        "image_embedding_dimension":768,"stored_dtype":"torch.float32; raw, not unit-normalized",
        "encoding_batch_size":8,"augmentation":None}


def validate_feature_cache(cache, split, table, config):
    expected=expected_cache_spec(split,table,config)
    stored_spec = cache.get("spec", {})
    comparable_spec = dict(stored_spec)
    if "native_transform" in comparable_spec:
        comparable_spec["native_transform"] = canonical_native_preprocess_repr(comparable_spec["native_transform"])
    # Validate the original raw digest before normalizing the legacy in-process repr.
    if (cache.get("schema") != "wp3b-native-clip-cache-v1" or comparable_spec != expected
            or cache.get("spec_sha256") != json_hash(stored_spec)):
        raise ValueError("Cache source/split/model/prompt/preprocessing contract mismatch.")
    if set(cache.get("folds",{})) != {"fit","val_seen","val_unseen"}:
        raise ValueError("Only the approved three training-derived folds are allowed.")
    provenance=cache["model_provenance"]
    if (provenance["model"] != "ViT-L/14" or provenance["revision"] != CLIP_REVISION
            or provenance["artifact"]["sha256"] != expected["model_sha256"]
            or canonical_native_preprocess_repr(provenance["native_preprocess"]) != expected["native_transform"]
            or provenance["trainable_parameters"] != 0):
        raise ValueError("Frozen cache model provenance changed.")
    names=[]
    for role in ("fit","val_seen","val_unseen"):
        frame=getattr(split,role);part=cache["folds"][role]
        labels=torch.tensor(frame[["crop_id","disease_id","composition_id"]].to_numpy(),dtype=torch.int64)
        x=part["features"]
        if (part["filenames"] != frame.image_name.tolist() or part["labels"].dtype != torch.int64
                or not torch.equal(part["labels"].cpu(),labels)
                or part["labels_sha256"] != tensor_sha256(labels)
                or x.shape != (len(frame),768) or x.dtype != torch.float32 or x.requires_grad
                or not torch.isfinite(x).all() or (torch.linalg.vector_norm(x,dim=1)<=0).any()
                or part["features_sha256"] != tensor_sha256(x)):
            raise ValueError("Cache row/tensor/label fingerprint changed: "+role)
        names.extend(part["filenames"])
    hashes=cache["image_hashes"]
    if (len(names)!=8000 or len(set(names))!=8000 or set(hashes)!=set(names)
            or any(not isinstance(v,str) or len(v)!=64 or any(c not in "0123456789abcdef" for c in v) for v in hashes.values())
            or cache["image_hashes_sha256"] != json_hash(hashes)):
        raise ValueError("Cache image identities/content provenance changed.")
    direct=cache["direct"]
    if (direct.shape!=(323,768) or direct.dtype!=torch.float64 or direct.requires_grad
            or not torch.isfinite(direct).all() or (torch.linalg.vector_norm(direct,dim=1)<=0).any()
            or cache["direct_sha256"]!=tensor_sha256(direct)):
        raise ValueError("Cache direct-text bank changed.")
    return True


def verify_cache_image_files(cache, config):
    """Re-hash only validated development filenames; never scan an image directory."""
    directory = Path(config["image_directory"]).resolve()
    for role in ("fit", "val_seen", "val_unseen"):
        for name in cache["folds"][role]["filenames"]:
            if (not isinstance(name, str) or Path(name).name != name
                    or any(character in name for character in ("/", "\\", ":"))):
                raise ValueError("Unsafe cached development filename.")
            path = (directory / name).resolve()
            if path.parent != directory or not path.is_file():
                raise ValueError("Missing/redirected approved development image: " + name)
            if sha256_file(path) != cache["image_hashes"][name]:
                raise ValueError("Current development image bytes differ from the cache: " + name)
    return True


def load_feature_cache(path, expected_sha256, split, table, config):
    path=require_external_path(path)
    if not path.is_file() or sha256_file(path)!=expected_sha256:
        raise ValueError("Missing cache or full artifact SHA-256 mismatch.")
    cache=torch.load(path,map_location="cpu",weights_only=True)
    validate_feature_cache(cache,split,table,config)
    verify_cache_image_files(cache,config)
    return cache


class CachedFeatureBackbone(torch.nn.Module):
    """Frozen identity boundary over previously verified raw CLIP embeddings."""
    def encode_image(self, features):
        if (features.ndim!=2 or features.shape[1]!=768 or not len(features)
                or not torch.isfinite(features).all()
                or (torch.linalg.vector_norm(features.float(),dim=1)<=0).any()):
            raise ValueError("Expected nonzero finite raw CLIP features [B,768].")
        return features.detach().float()


class CachedFeatureDataset(Dataset):
    def __init__(self,split,role,cache):
        if role not in ("fit","val_seen","val_unseen"):
            raise ValueError("Only training-derived cache roles are supported.")
        frame=getattr(split,role)
        self.role=role;self.frame=frame.copy();self.part=cache["folds"][role]
        labels=torch.tensor(frame[["crop_id","disease_id","composition_id"]].to_numpy(),dtype=torch.int64)
        if self.part["filenames"]!=frame.image_name.tolist() or not torch.equal(self.part["labels"],labels):
            raise ValueError("Cache dataset does not match the exact approved role.")
    def __len__(self):
        return len(self.frame)
    def __getitem__(self,index):
        c,d,p=self.part["labels"][index].tolist()
        return self.part["features"][index],{"crop":c,"disease":d,"composition":p,
                                          "image_name":self.part["filenames"][index]}


def verify_cached_frozen_reference(cache, split):
    """Owner-only cached scoring; stops on any historical Top-5 rank discrepancy."""
    path=Path(WORKSPACE_DIRECTORY)/"experiments/wp3b-m3/20261104T094921Z_365ee1b25a43/adapter_epoch_predictions.csv"
    reference=pd.read_csv(path)
    reference=reference.loc[reference.epoch==0]
    for role in ("val_seen","val_unseen"):
        part=cache["folds"][role];expected=reference.loc[reference.fold==role]
        if expected.image_name.tolist()!=part["filenames"]:
            raise ValueError("Historical frozen reference order differs.")
        ranks=rank_global_ids(cosine_candidate_scores(part["features"],cache["direct"]),split.candidate_ids)[:,:5]
        expected=torch.tensor(expected[["control_rank_"+str(k) for k in range(1,6)]].to_numpy())
        if not torch.equal(ranks,expected):
            raise ValueError("Cached/native frozen ranks differ. Preserve cache and investigate; no tolerance is silently applied.")
    return True


## 10. Recorded-result helpers and unchanged disease-safety gate

Retain original per-pair/healthy/diseased diagnostics and gate thresholds. Preserve failed seed42 explicitly. Higher H@1 alone is not evidence that disease transfer improved.

In [ ]:
def history_row(epoch, train, validation):
    seen, unseen = validation["val_seen"], validation["val_unseen"]
    row = selection_record(epoch, seen, unseen)
    row = {k: v for k, v in row.items() if k != "selection_policy"}
    for role, result in validation.items():
        for key in ("loss", "crop_loss", "disease_loss", "identity_loss", "elapsed_seconds",
                    "peak_allocated_bytes", "peak_reserved_bytes"):
            row[role + "_" + key] = result[key]
    for key in ("loss", "crop_loss", "disease_loss", "identity_loss", "elapsed_seconds",
                "peak_allocated_bytes", "peak_reserved_bytes"):
        row["train_" + key] = None if train is None else train[key]
    row["train_sample_count"] = None if train is None else train["sample_count"]
    row["train_conditional_loss"] = None if train is None else train["conditional_loss"]
    return row


def new_checkpoint_path(run_directory, epoch):
    return Path(run_directory) / ("adapter_epoch_" + str(epoch).zfill(3) + "_" + uuid.uuid4().hex[:8] + ".pt")


def prediction_rows(validation, epoch):
    rows = []
    for role, result in validation.items():
        for i, name in enumerate(result["filenames"]):
            row = {"epoch": int(epoch), "fold": role, "image_name": name,
                   "truth_global_id": int(result["truths"][i])}
            row.update({"rank_" + str(k+1): int(v) for k, v in enumerate(result["ranking"][i, :5])})
            row.update({"control_rank_" + str(k+1): int(v) for k, v in enumerate(result["control_ranking"][i, :5])})
            rows.append(row)
    return rows


def diagnose_predictions(predictions, split):
    """Counts from complete saved development ranks only; no image/model access."""
    expected = pd.concat([split.val_seen.assign(fold="val_seen"),
                          split.val_unseen.assign(fold="val_unseen")], ignore_index=True)
    if (predictions.duplicated(["fold", "image_name"]).any() or len(predictions) != len(expected)
            or set(zip(predictions.fold, predictions.image_name)) != set(zip(expected.fold, expected.image_name))):
        raise ValueError("Require each development filename exactly once; missing rows are not dropped.")
    frame = expected.merge(predictions, on=["fold", "image_name"], validate="one_to_one")
    if not frame.composition_id.eq(frame.truth_global_id).all():
        raise ValueError("Saved truths differ from the approved training-derived manifest.")
    pair = dict(zip(split.candidate_ids, split.candidate_pairs))
    for column in ["truth_global_id"] + ["rank_"+str(k) for k in range(1,6)]:
        values = frame[column]
        if values.dtype.kind not in "iu" or not set(values) <= set(pair):
            raise ValueError("Prediction values must be original integer global IDs.")
    if any(len(set(row)) != 5 for row in frame[["rank_"+str(k) for k in range(1,6)]].itertuples(index=False,name=None)):
        raise ValueError("Duplicate ranked candidate IDs.")
    healthy = {int(row.disease_id) for row in split.fit.itertuples()
               if normalize_label(row.disease_name) == "healthy"}
    predicted = frame.rank_1.map(pair)
    frame["pred_crop"] = predicted.map(lambda value:value[0])
    frame["pred_disease"] = predicted.map(lambda value:value[1])
    frame["correct1"] = frame.rank_1.eq(frame.composition_id)
    frame["correct5"] = frame[["rank_"+str(k) for k in range(1,6)]].eq(frame.composition_id,axis=0).any(axis=1)
    frame["true_healthy"] = frame.disease_id.isin(healthy)
    frame["pred_healthy"] = frame.pred_disease.isin(healthy)
    def counts(part):
        return {"count":len(part),"correct1":int(part.correct1.sum()),"correct5":int(part.correct5.sum()),
            "top1":float(part.correct1.mean()) if len(part) else None,
            "top5":float(part.correct5.mean()) if len(part) else None,
            "crop_correct":int(part.pred_crop.eq(part.crop_id).sum()),
            "disease_correct":int(part.pred_disease.eq(part.disease_id).sum()),
            "predicted_healthy":int(part.pred_healthy.sum()),
            "healthy_count":int(part.true_healthy.sum()),
            "healthy_correct":int((part.correct1 & part.true_healthy).sum()),
            "diseased_count":int((~part.true_healthy).sum()),
            "diseased_correct":int((part.correct1 & ~part.true_healthy).sum()),
            "false_healthy":int((part.pred_healthy & ~part.true_healthy).sum()),
            "predicted_fitting_composition":int(part.rank_1.isin(split.fit.composition_id.unique()).sum())}
    result = {role:counts(part) for role,part in frame.groupby("fold")}
    result["per_pair"] = [{"fold":role,"global_composition_id":int(key),**counts(part)}
                          for (role,key),part in frame.groupby(["fold","composition_id"])]
    result["per_crop"] = [{"fold":role,"crop":crop,**counts(part)}
                          for (role,crop),part in frame.groupby(["fold","crop_name"])]
    result["per_disease"] = [{"fold":role,"disease_id":int(disease),**counts(part)}
                             for (role,disease),part in frame.groupby(["fold","disease_id"])]
    result["focus_crops"] = [{"fold":role,"crop":crop,**counts(part)}
                             for (role,crop),part in frame[frame.crop_name.isin(["tomato","cassava"])].groupby(["fold","crop_name"])]
    return result


def compare_pilot_summaries(control, intervention):
    if control["protocol"] != intervention["protocol"]:
        raise ValueError("Paired candidate/source/split/model/code protocols differ.")
    if control["config"]["residual_cap"] is not None or intervention["config"]["residual_cap"] != .2:
        raise ValueError("Compare only the unbounded control and rho=.20 intervention.")
    stripped=lambda cfg:{k:v for k,v in cfg.items() if k not in ("residual_cap","output_directory")}
    if stripped(control["config"])!=stripped(intervention["config"]):
        raise ValueError("Both arms require identical loss, cache, optimizer and budgets.")
    for summary in (control,intervention):
        if [row["epoch"] for row in summary["history"]]!=list(range(6)):
            raise ValueError("Both five-epoch budgets plus epoch zero must be complete.")
        if any(row.get("train_sample_count")!=4868 for row in summary["history"][1:]):
            raise ValueError("Each fitting epoch must process all 4868 samples.")
        for role,expected in (("val_seen",1238),("val_unseen",1894)):
            d=summary["diagnostics"][role];key="seen_top1" if role=="val_seen" else "unseen_top1"
            if d["count"]!=expected or summary["selected"][key]!=d["correct1"]/expected:
                raise ValueError("Metric and full processed counts disagree.")
        if summary["selected"]["h1"]!=harmonic_mean(summary["selected"]["seen_top1"],summary["selected"]["unseen_top1"]):
            raise ValueError("Stored H@1 disagrees with fold accuracies.")
    a=control["diagnostics"]["val_unseen"];b=intervention["diagnostics"]["val_unseen"]
    gain=intervention["selected"]["h1"]>max(control["selected"]["h1"],.1464606922871855)
    disease=b["diseased_correct"]>max(a["diseased_correct"],71)
    false_healthy=b["false_healthy"]<min(a["false_healthy"],490)
    return {"strict_H1_gain":gain,"improved_unseen_diseased_hits":disease,
        "reduced_false_healthy":false_healthy,"frozen_diseased_79_recovered":b["diseased_correct"]>=79,
        "supports_exploratory_success":gain and disease and false_healthy and b["diseased_correct"]>=79,
        "interpretation":"One fixed development pilot; no significance or final-test claim."}


## 11. Predeclared seed contracts and arm initialization

In [ ]:
def validate_replication_config(config):
    if "seed" in config:
        raise ValueError("Use explicit split_seed and training_seed; legacy seed is ambiguous.")
    if type(config.get("split_seed")) is not int or config["split_seed"] != 42:
        raise ValueError("The accepted development split seed is fixed at42.")
    seeds = config.get("training_seeds")
    if not isinstance(seeds, list) or seeds != [43, 44] or any(type(v) is not int for v in seeds):
        raise ValueError("Exactly training seeds43 and44 in that order are predeclared.")
    fixed = {"model": "ViT-L/14", "batch_size": 8, "text_batch_size": 32, "num_workers": 0,
             "epochs": 5, "learning_rate": 1e-4, "weight_decay": 1e-4, "conditional_weight": 1.0}
    if any(config.get(k) != v or isinstance(config.get(k), bool) for k, v in fixed.items()):
        raise ValueError("Replication fixes the architecture, objective, optimizer and five-epoch budget.")
    for name in ("batch_size", "text_batch_size", "num_workers", "epochs"):
        if type(config.get(name)) is not int:
            raise ValueError("Epoch/batch/worker budgets must be integers, not floats or booleans.")
    if "training_seed" in config and (type(config["training_seed"]) is not int or config["training_seed"] not in (43, 44)):
        raise ValueError("Only the declared integer training seeds43 and44 are allowed.")
    if "residual_cap" in config and (config["residual_cap"] is not None and
            (type(config["residual_cap"]) not in (int, float) or config["residual_cap"] != .2)):
        raise ValueError("Only an unbounded control or cap0.20 is allowed.")
    if torch.device(config["device"]).type not in ("cpu", "cuda"):
        raise ValueError("CPU/CUDA device required.")
    require_external_path(config["output_directory"])
    require_external_path(config["feature_cache_path"])
    fingerprint = config.get("feature_cache_sha256")
    if (not isinstance(fingerprint, str) or len(fingerprint) != 64
            or any(c not in "0123456789abcdef" for c in fingerprint)):
        raise ValueError("Explicit full cache SHA-256 required.")
    return True


def initialize_replication_adapter(training_seed, residual_cap):
    if type(training_seed) is not int or training_seed not in (43, 44):
        raise ValueError("Only the two declared training seeds are supported.")
    random.seed(training_seed)
    np.random.seed(training_seed)
    torch.manual_seed(training_seed)
    return ResidualVisualAdapter(residual_cap)


def make_replication_loader(split, role, config, cache):
    verify_split_identity(split)
    validate_replication_config(config)
    if type(config.get("training_seed")) is not int or config["training_seed"] not in (43, 44):
        raise ValueError("A declared training_seed is required for the fitting generator.")
    dataset = CachedFeatureDataset(split, role, cache)
    return DataLoader(dataset, batch_size=config["batch_size"], shuffle=role == "fit", drop_last=False,
        num_workers=0, pin_memory=torch.device(config["device"]).type == "cuda",
        generator=torch.Generator().manual_seed(config["training_seed"]))


def make_replication_contract(split, table, config):
    validate_replication_config(config)
    seed = config.get("training_seed")
    if type(seed) is not int or seed not in (43, 44):
        raise ValueError("Declare exactly training seed43 or44; split seed stays42.")
    for name in ("batch_size", "text_batch_size", "epochs"):
        if type(config[name]) is not int or config[name] < 1:
            raise ValueError("Positive integer budgets required.")
    if type(config["num_workers"]) is not int or config["num_workers"] < 0:
        raise ValueError("num_workers must be nonnegative.")
    for name in ("learning_rate", "weight_decay"):
        value = config[name]
        if isinstance(value, bool) or not isinstance(value, Real) or not math.isfinite(value):
            raise ValueError("Finite optimizer settings required.")
        if value < 0 or (name == "learning_rate" and value == 0):
            raise ValueError("Invalid optimizer setting.")
    if torch.device(config["device"]).type not in ("cpu", "cuda"):
        raise ValueError("Only CPU/CUDA devices are supported.")
    for key in ("image_directory", "weight_directory", "output_directory"):
        if not Path(config[key]).is_absolute():
            raise ValueError("Explicit absolute WSL paths required.")
    if config["epochs"] > 5:
        raise ValueError("The initial pilot is capped at five epochs.")
    fixed = {"batch_size": 8, "text_batch_size": 32, "num_workers": 0, "epochs": 5,
             "learning_rate": 1e-4, "weight_decay": 1e-4}
    if any(config.get(k) != v for k, v in fixed.items()):
        raise ValueError("The paired pilot fixes batch, optimizer and five-epoch budget.")
    if type(config.get("conditional_weight")) not in (int, float) or config["conditional_weight"] not in (0., 1.):
        raise ValueError("Select only the declared control (0) or intervention (1).")
    require_external_path(config["output_directory"])
    if config["conditional_weight"] != 1.0 or config.get("residual_cap") not in (None, .2):
        raise ValueError("M4 keeps M3 loss fixed and compares only unbounded versus rho=.20.")
    if isinstance(config.get("residual_cap"), bool):
        raise ValueError("Residual cap cannot be Boolean.")
    if not config.get("feature_cache_path") or not Path(config["feature_cache_path"]).is_absolute():
        raise ValueError("Declare an explicit verified shared-cache path.")
    require_external_path(config["feature_cache_path"])
    fingerprint = config.get("feature_cache_sha256")
    if not isinstance(fingerprint, str) or len(fingerprint) != 64 or any(x not in "0123456789abcdef" for x in fingerprint):
        raise ValueError("An explicit verified cache SHA-256 is required.")
    expected_table = build_prompt_table(split)
    if not table.equals(expected_table):
        raise ValueError("Candidate/prompt ordering changed.")
    prompt_hash = json_hash({"policy": PROMPT_POLICY, "rows": table.to_dict("records")})
    if prompt_hash != EXPECTED_DIRECT_PROMPT_SHA256:
        raise ValueError("The accepted direct prompt policy changed.")
    if LOSS_SPEC != {"crop_weight": .5, "disease_weight": .5, "identity_weight": 1.,
                     "temperature": .07, "identity": "mean sample squared L2 between unit adapted and unit frozen features"}:
        raise ValueError("M2 has one fixed predeclared objective.")
    return {"schema": "wp3b-m5-replication-contract-v1", "split": verify_split_identity(split),
        "fitting_composition_ids": sorted(int(x) for x in split.fit.composition_id.unique()),
        "candidate_ids": list(split.candidate_ids), "candidate_pairs": [list(x) for x in split.candidate_pairs],
        "component_ids": {k: [int(x) for x in sorted(table[k + "_id"].unique())] for k in ("crop", "disease")},
        "component_names": {k: [[int(i), str(n)] for i, n in sorted(split.mappings[k + "_id_to_name"].items())]
                            for k in ("crop", "disease")},
        "direct_prompt_policy": dict(PROMPT_POLICY), "direct_prompt_sha256": prompt_hash,
        "direct_prompt_records": table.to_dict("records"), "model": {"name": "ViT-L/14",
            "revision": CLIP_REVISION, **weight_spec("ViT-L/14")},
        "adapter": {**ADAPTER_SPEC, "residual_cap": config["residual_cap"]}, "residual_bound": dict(RESIDUAL_CAP_SPEC), "component_prototypes": dict(PROTOTYPE_SPEC),
        "conditional_objective": dict(M3_SPEC), "loss": dict(LOSS_SPEC), "preprocessing": dict(NATIVE_PREPROCESSING),
        "hyperparameters": dict(config), "optimizer": "AdamW; adapter only; fixed LR; no AMP/scheduler",
        "gradient_norm_clip": 1., "selection_policy": dict(SELECTION_POLICY),
        "code": {"WP3A_parent": WP3A_PARENT, "M4_review_parent": REPLICATION_PARENT, "algorithm_sha256": notebook_algorithm_hash()},
        "environment": {"python": platform.python_version(), "torch": str(torch.__version__),
                        "torchvision": str(torchvision.__version__)}}


## 12. Four-arm owner runner and exclusive recorded outputs

All four contracts are written before the first optimizer step. Both arms of each seed start with identical parameters and loader order. Any failure or interruption is recorded and re-raised; partial groups cannot produce a completed comparison. No automatic retry/resumption.

In [ ]:
def write_exclusive_json(path, value):
    path = require_external_path(path)
    with path.open("x", encoding="utf-8") as handle:
        json.dump(value, handle, indent=2, allow_nan=False)
    return str(path)


def verify_replication_cache(cache, split, table, config):
    validate_replication_config(config)
    validate_feature_cache(cache, split, table, config)
    if sha256_file(require_external_path(config["feature_cache_path"])) != config["feature_cache_sha256"]:
        raise ValueError("Pinned cache artifact changed.")
    verify_cache_image_files(cache, config)
    verify_cached_frozen_reference(cache, split)
    return True


def evaluate_replication_folds(split, model, adapter, state, config, cache):
    return {role: validate_adapter(model, adapter,
                make_replication_loader(split, role, config, cache), state, getattr(split, role),
                split.candidate_ids, config["device"])
            for role in ("val_seen", "val_unseen")}


def verify_replication_state(split, table, config, cache, contract, directory, state):
    if make_replication_contract(split, table, config) != contract:
        raise ValueError("Declared source/configuration/code changed.")
    declaration = json.loads((Path(directory) / "declaration.json").read_text())
    if declaration["contract"] != contract or declaration["contract_sha256"] != json_hash(contract):
        raise ValueError("Arm declaration changed.")
    verify_replication_cache(cache, split, table, config)
    validate_text_state(state, contract)
    return True


def export_replication_results(run_directory, contract, pilot, split):
    directory = require_external_path(run_directory)
    targets = [directory / name for name in ("adapter_history.csv", "adapter_selected_predictions.csv", "adapter_summary.json", "adapter_epoch_predictions.csv")]
    if any(p.exists() for p in targets):
        raise FileExistsError("Results already exported; preserve them and use a fresh run.")
    rows = []
    for role, result in pilot["best_validation"].items():
        for i, name in enumerate(result["filenames"]):
            row = {"fold": role, "image_name": name, "truth_global_id": int(result["truths"][i])}
            row.update({"rank_" + str(k + 1): int(v) for k, v in enumerate(result["ranking"][i, :5])})
            rows.append(row)
    summary = {"contract_sha256": json_hash(contract), "selected": pilot["best_record"],
        "baseline": pilot["baseline_record"], "checkpoint": pilot["best_checkpoint"],
        "strict_H1_gain_over_identity": pilot["strict_H1_gain_over_identity"],
        "history": pilot["history"], "diagnostics": diagnose_predictions(pd.DataFrame(rows), split),
        "config": contract["hyperparameters"], "protocol": {key: contract[key] for key in ("split", "candidate_ids", "candidate_pairs",
            "direct_prompt_sha256", "model", "preprocessing", "code", "conditional_objective")},
        "optimizer_resumption_supported": False,
        "interpretation": "A loss-only tie-break does not establish an H@1 improvement; no significance claim."}
    serialized = json.dumps(summary, indent=2, allow_nan=False)
    pd.DataFrame(pilot["history"]).to_csv(targets[0], index=False, mode="x")
    pd.DataFrame(rows).to_csv(targets[1], index=False, mode="x")
    with targets[2].open("x", encoding="utf-8") as handle:
        handle.write(serialized)
    pd.DataFrame(pilot["epoch_predictions"]).to_csv(targets[3], index=False, mode="x")
    return [str(p) for p in targets]


### 12.1 Execute the predeclared arms only on explicit invocation

In [ ]:
def run_replication_arm(split, table, config, cache, contract, directory, adapter):
    """Owner-only five epochs. The visible accepted epoch functions perform every update."""
    model = CachedFeatureBackbone().eval().to(config["device"])
    require_frozen_clip(model)
    adapter = adapter.to(config["device"])
    if torch.count_nonzero(adapter.up.weight) or torch.count_nonzero(adapter.up.bias):
        raise ValueError("A fresh identity adapter is required; recovery-state reuse is prohibited.")
    prototypes = component_prototypes(cache["direct"], table, contract["fitting_composition_ids"])
    state = seal_text_state(cache["direct"], prototypes, contract)
    verify_replication_state(split, table, config, cache, contract, directory, state)
    baseline = evaluate_replication_folds(split, model, adapter, state, config, cache)
    if any(not torch.equal(v["ranking"], v["control_ranking"]) for v in baseline.values()):
        raise ValueError("Identity epoch0 does not reproduce frozen predictions.")
    baseline_record = selection_record(0, baseline["val_seen"], baseline["val_unseen"])
    checkpoint = save_adapter_checkpoint(new_checkpoint_path(directory, 0), adapter, state, contract, baseline_record)
    best, best_validation, best_checkpoint = baseline_record, baseline, checkpoint
    history = [history_row(0, None, baseline)]
    predictions = prediction_rows(baseline, 0)
    loader = make_replication_loader(split, "fit", config, cache)
    optimizer = make_adapter_optimizer(adapter, config)
    for epoch in range(1, 6):
        training = train_adapter_epoch(model, adapter, loader, optimizer, state, split.fit, config["device"])
        validation = evaluate_replication_folds(split, model, adapter, state, config, cache)
        record = selection_record(epoch, validation["val_seen"], validation["val_unseen"])
        history.append(history_row(epoch, training, validation))
        predictions.extend(prediction_rows(validation, epoch))
        if better_selection(record, best):
            best, best_validation = record, validation
            best_checkpoint = save_adapter_checkpoint(new_checkpoint_path(directory, epoch), adapter, state, contract, record)
        print({"training_seed": config["training_seed"], "cap": config["residual_cap"],
               "epoch": epoch, "training_loss": training["loss"], "validation": record,
               "selected_epoch": best["epoch"], "selected_checkpoint": best_checkpoint})
    pilot = {"history": history, "epoch_predictions": predictions, "best_record": best,
             "best_validation": best_validation, "best_checkpoint": best_checkpoint,
             "baseline_record": baseline_record, "strict_H1_gain_over_identity": best["h1"] > baseline_record["h1"]}
    export_replication_results(directory, contract, pilot, split)
    summary_path = Path(directory) / "adapter_summary.json"
    summary = json.loads(summary_path.read_text())
    validate_replication_summary(summary, split)
    validate_recorded_predictions(directory, summary, split)
    artifacts = {file.name: sha256_file(file) for file in sorted(Path(directory).iterdir()) if file.is_file()}
    return {"training_seed": config["training_seed"], "arm": "control" if config["residual_cap"] is None else "bounded",
            "directory": str(directory), "summary_path": str(summary_path),
            "summary_sha256": sha256_file(summary_path), "artifacts": artifacts}


def run_replications(split, table, config, cache):
    """Exactly four predeclared owner-run arms; an interrupted attempt is never resumed."""
    global REPLICATION_STARTED
    if REPLICATION_STARTED:
        raise RuntimeError("This kernel already started a group. Preserve it; use a fresh explicit attempt.")
    validate_replication_config(config)
    if torch.device(config["device"]).type == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("The declared CUDA device is unavailable; do not silently change devices.")
    verify_replication_cache(cache, split, table, config)
    arms = []
    for seed in (43, 44):
        for label, cap in (("control", None), ("bounded", .2)):
            arm_config = {**config, "training_seed": seed, "residual_cap": cap}
            arms.append({"training_seed": seed, "arm": label, "config": arm_config,
                         "contract": make_replication_contract(split, table, arm_config)})
    # No model optimization has occurred; all four configurations are now fixed.
    REPLICATION_STARTED = True
    root = require_external_path(config["output_directory"])
    root.mkdir(parents=True, exist_ok=True)
    directory = root / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:12])
    directory.mkdir(exist_ok=False)
    records, initial_hashes = [], {}
    try:
        declaration_arms = []
        for item in arms:
            folder = directory / ("seed_" + str(item["training_seed"]) + "_" + item["arm"])
            folder.mkdir(exist_ok=False)
            item["directory"] = str(folder)
            write_exclusive_json(folder / "declaration.json", {"contract": item["contract"],
                "contract_sha256": json_hash(item["contract"]), "predeclared_utc": datetime.now(timezone.utc).isoformat()})
            table.to_csv(folder / "direct_prompts.csv", index=False, mode="x")
            declaration_arms.append({"training_seed": item["training_seed"], "arm": item["arm"],
                "directory": str(folder), "contract": item["contract"], "contract_sha256": json_hash(item["contract"])})
        group = {"schema": "wp3b-m5-group-v1", "config": dict(config), "arms": declaration_arms,
                 "historical_reference": HISTORICAL_REFERENCE, "code_sha256": notebook_algorithm_hash(),
                 "selection_policy": dict(SELECTION_POLICY), "purpose": "descriptive seed replication; no significance claim"}
        write_exclusive_json(directory / "replication_declaration.json", group)
        for item in arms:
            seed, cap = item["training_seed"], item["config"]["residual_cap"]
            adapter = initialize_replication_adapter(seed, cap)
            initial_hash = json_hash({k: tensor_sha256(v) for k, v in adapter.state_dict().items()})
            if seed in initial_hashes and initial_hashes[seed] != initial_hash:
                raise ValueError("The paired arms do not start from identical initialization.")
            initial_hashes[seed] = initial_hash
            write_exclusive_json(Path(item["directory"]) / "initial_state.json",
                                 {"training_seed": seed, "split_seed": 42, "sha256": initial_hash})
            result = run_replication_arm(split, table, item["config"], cache, item["contract"],
                                        item["directory"], adapter)
            if (result["training_seed"], result["arm"]) != (seed, item["arm"]):
                raise ValueError("Arm result identity differs from its declaration.")
            validate_replication_summary(json.loads(Path(result["summary_path"]).read_text()), split)
            records.append(result)
        write_exclusive_json(directory / "completed.json", {"schema": "wp3b-m5-completed-v1",
            "declaration_sha256": sha256_file(directory / "replication_declaration.json"), "records": records,
            "completed_utc": datetime.now(timezone.utc).isoformat()})
    except BaseException as exc:
        write_exclusive_json(directory / "failed.json", {"error_type": type(exc).__name__,
            "message": str(exc), "completed_arms": [{"training_seed": x["training_seed"], "arm": x["arm"]} for x in records],
            "interpretation": "Incomplete group: no paired effectiveness or aggregate success claim."})
        raise
    print({"completed_group": str(directory), "new_arms": len(records), "historical_seed42_rerun": False})
    return {"group_directory": str(directory), "records": records}


## 13. Historical provenance and descriptive replication reporting

Verify seed42 artifacts by recorded SHA-256 and full prediction records. New seeds need the same complete counts and selection policy. Report all six runs, paired seed differences, descriptive mean/sample standard deviation and per-pair disease/healthy results. Three training seeds are not independent dataset samples and do not establish statistical significance.

In [ ]:
def validate_replication_summary(summary, split, historical=False):
    cfg, protocol = summary["config"], summary["protocol"]
    if historical:
        if type(cfg.get("seed")) is not int or cfg["seed"] != 42 or "training_seed" in cfg:
            raise ValueError("Historical reference must remain the original seed42 experiment.")
    else:
        validate_replication_config(cfg)
        if type(cfg.get("training_seed")) is not int or cfg["training_seed"] not in (43, 44):
            raise ValueError("Unexpected new training seed.")
    fixed = {"model": "ViT-L/14", "batch_size": 8, "epochs": 5, "conditional_weight": 1.,
             "learning_rate": 1e-4, "weight_decay": 1e-4, "num_workers": 0}
    if any(cfg.get(k) != v for k, v in fixed.items()) or cfg.get("residual_cap") not in (None, .2):
        raise ValueError("Recorded architecture/objective/optimizer/cap/budget changed.")
    if cfg.get("feature_cache_sha256") != CONFIG["feature_cache_sha256"]:
        raise ValueError("Comparison requires the same originally pinned frozen cache.")
    expected = {"split": verify_split_identity(split), "candidate_ids": list(split.candidate_ids),
        "candidate_pairs": [list(p) for p in split.candidate_pairs],
        "direct_prompt_sha256": EXPECTED_DIRECT_PROMPT_SHA256,
        "model": {"name": "ViT-L/14", "revision": CLIP_REVISION, **weight_spec("ViT-L/14")},
        "preprocessing": NATIVE_PREPROCESSING, "conditional_objective": M3_SPEC}
    if any(protocol.get(k) != value for k, value in expected.items()):
        raise ValueError("Recorded split/candidate/prompt/model/native preprocessing/objective changed.")
    history = summary["history"]
    if len(history) != 6 or [row["epoch"] for row in history] != list(range(6)):
        raise ValueError("Exactly epoch0 and all five fitting epochs are required.")
    for row in history:
        if type(row["epoch"]) is not int:
            raise ValueError("Epoch identity must be integer.")
        for key, count in (("seen_count", 1238), ("unseen_count", 1894)):
            if type(row.get(key)) is not int or row[key] != count:
                raise ValueError("Incomplete validation-fold denominator.")
        if row["epoch"] and (type(row.get("train_sample_count")) is not int or row["train_sample_count"] != 4868):
            raise ValueError("Incomplete fitting-epoch budget.")
        for key in ("seen_top1", "seen_top5", "unseen_top1", "unseen_top5", "h1", "h5"):
            value = row[key]
            if isinstance(value, bool) or not isinstance(value, Real) or not math.isfinite(value) or not 0 <= value <= 1:
                raise ValueError("Non-finite or out-of-range recorded accuracy.")
        if row["seen_top5"] < row["seen_top1"] or row["unseen_top5"] < row["unseen_top1"]:
            raise ValueError("Top5 cannot be below Top1.")
        if (row["h1"] != harmonic_mean(row["seen_top1"], row["unseen_top1"])
                or row["h5"] != harmonic_mean(row["seen_top5"], row["unseen_top5"])):
            raise ValueError("Recorded harmonic means disagree with fold metrics.")
        for key in ("val_seen_loss", "val_unseen_loss"):
            value = row[key]
            if isinstance(value, bool) or not isinstance(value, Real) or not math.isfinite(value) or value < 0:
                raise ValueError("Recorded cross-entropy objectives must be finite and nonnegative.")
        combined = (1238 * row["val_seen_loss"] + 1894 * row["val_unseen_loss"]) / 3132
        if not math.isfinite(combined) or combined != row["combined_validation_loss"]:
            raise ValueError("Recorded sample-weighted validation loss disagrees.")
    best = min(history, key=lambda row: (-row["h1"], row["combined_validation_loss"], row["epoch"]))
    selected = summary["selected"]
    if type(selected.get("epoch")) is not int or any(type(selected.get(k)) is not int for k in ("seen_count", "unseen_count")):
        raise ValueError("Selected epoch and fold counts must retain integer provenance.")
    for key in ("epoch", "h1", "h5", "seen_top1", "seen_top5", "unseen_top1", "unseen_top5",
                "seen_count", "unseen_count", "combined_validation_loss"):
        if selected.get(key) != best[key]:
            raise ValueError("Selected checkpoint differs from the predeclared full-precision policy.")
    if selected.get("selection_policy") != SELECTION_POLICY:
        raise ValueError("Checkpoint-selection policy changed.")
    for role, count, healthy, prefix in (("val_seen", 1238, 238, "seen"), ("val_unseen", 1894, 398, "unseen")):
        d = summary["diagnostics"][role]
        for key, expected_count in (("count", count), ("healthy_count", healthy), ("diseased_count", count - healthy)):
            if type(d.get(key)) is not int or d[key] != expected_count:
                raise ValueError("Diagnostic sample support changed.")
        for key in ("correct1", "correct5", "healthy_correct", "diseased_correct", "false_healthy"):
            value = d[key]
            limit = healthy if key == "healthy_correct" else count - healthy if key in ("diseased_correct", "false_healthy") else count
            if type(value) is not int or not 0 <= value <= limit:
                raise ValueError("Invalid diagnostic count.")
        if (d["correct1"] != d["healthy_correct"] + d["diseased_correct"] or d["correct5"] < d["correct1"]
                or selected[prefix + "_top1"] != d["correct1"] / count
                or selected[prefix + "_top5"] != d["correct5"] / count):
            raise ValueError("Selected metrics and subgroup hit counts disagree.")
    return True


def validate_recorded_predictions(directory, summary, split):
    """Re-count saved rows, not model inference; reject missing/reordered/invalid records."""
    directory = Path(directory)
    predictions = pd.read_csv(directory / "adapter_epoch_predictions.csv")
    if (predictions.epoch.dtype.kind not in "iu" or sorted(predictions.epoch.unique().tolist()) != list(range(6))
            or len(predictions) != 6 * 3132):
        raise ValueError("Require all18792 complete epoch-level prediction rows.")
    epochs = []
    for epoch, rows in predictions.groupby("epoch", sort=True):
        for role in ("val_seen", "val_unseen"):
            part, expected = rows.loc[rows.fold == role], getattr(split, role)
            if part.image_name.tolist() != expected.image_name.tolist() or part.truth_global_id.tolist() != expected.composition_id.tolist():
                raise ValueError("Ordered prediction filename/truth manifest changed.")
        d = diagnose_predictions(rows, split)
        record = summary["history"][int(epoch)]
        for role, prefix in (("val_seen", "seen"), ("val_unseen", "unseen")):
            if record[prefix + "_top1"] != d[role]["top1"] or record[prefix + "_top5"] != d[role]["top5"]:
                raise ValueError("Saved predictions disagree with recorded metrics.")
            epochs.append({"epoch": int(epoch), "fold": role, **d[role],
                           "h1": record["h1"], "h5": record["h5"]})
    selected = pd.read_csv(directory / "adapter_selected_predictions.csv")
    chosen = predictions.loc[predictions.epoch == summary["selected"]["epoch"], selected.columns].reset_index(drop=True)
    if not chosen.equals(selected) or diagnose_predictions(selected, split) != summary["diagnostics"]:
        raise ValueError("Selected predictions/subgroup records disagree with the selected epoch.")
    return epochs


def verify_saved_arm(directory, summary, split, expected_contract=None):
    directory = require_external_path(directory)
    declaration = json.loads((directory / "declaration.json").read_text())
    contract = declaration["contract"]
    if (declaration["contract_sha256"] != json_hash(contract)
            or summary["contract_sha256"] != json_hash(contract)
            or expected_contract is not None and contract != expected_contract):
        raise ValueError("Saved arm contract fingerprint mismatch.")
    for key in summary["protocol"]:
        if summary["protocol"][key] != contract[key]:
            raise ValueError("Summary and declaration protocols disagree.")
    if summary["config"] != contract["hyperparameters"]:
        raise ValueError("Summary and declaration hyperparameters disagree.")
    checkpoint = Path(summary["checkpoint"]).resolve()
    if checkpoint.parent != directory or not checkpoint.is_file():
        raise ValueError("Selected checkpoint is outside its declared arm directory.")
    _, _, selected = load_adapter_checkpoint(checkpoint, contract)
    if selected != summary["selected"]:
        raise ValueError("Selected checkpoint metadata disagrees with results.")
    return validate_recorded_predictions(directory, summary, split)


### 13.1 Verify historical and new recorded inputs

In [ ]:
def load_historical_reference(split, table, config):
    records = []
    for label in ("control", "bounded"):
        reference = HISTORICAL_REFERENCE[label]
        directory = require_external_path(reference["directory"])
        for name, digest in reference["files"].items():
            path = (directory / name).resolve()
            if path.parent != directory or not path.is_file() or sha256_file(path) != digest:
                raise ValueError("Immutable seed42 artifact changed: " + name)
        summary = json.loads((directory / "adapter_summary.json").read_text())
        validate_replication_summary(summary, split, historical=True)
        epochs = verify_saved_arm(directory, summary, split)
        records.append({"training_seed": 42, "arm": label, "source": "historical_seed42",
                        "directory": str(directory), "summary": summary, "epochs": epochs})
    compare_pilot_summaries(records[0]["summary"], records[1]["summary"])
    return records


def read_completed_replication(directory, split, table, config):
    directory = require_external_path(directory)
    if (directory / "failed.json").exists() or not (directory / "completed.json").is_file():
        raise ValueError("An incomplete/failed group cannot produce a replication comparison.")
    completed = json.loads((directory / "completed.json").read_text())
    declaration = json.loads((directory / "replication_declaration.json").read_text())
    if (completed.get("schema") != "wp3b-m5-completed-v1" or declaration.get("schema") != "wp3b-m5-group-v1"
            or completed["declaration_sha256"] != sha256_file(directory / "replication_declaration.json")
            or declaration["config"] != config or declaration["code_sha256"] != notebook_algorithm_hash()
            or declaration["selection_policy"] != SELECTION_POLICY or declaration["historical_reference"] != HISTORICAL_REFERENCE):
        raise ValueError("Group configuration/source/code/history declaration changed.")
    expected = [(43, "control"), (43, "bounded"), (44, "control"), (44, "bounded")]
    if ([(x["training_seed"], x["arm"]) for x in completed["records"]] != expected
            or [(x["training_seed"], x["arm"]) for x in declaration["arms"]] != expected):
        raise ValueError("Require exactly four declared new-seed arms in the fixed order.")
    records, initial = [], {}
    for saved, declared in zip(completed["records"], declaration["arms"]):
        folder = require_external_path(saved["directory"])
        if folder.parent != directory or str(folder) != declared["directory"]:
            raise ValueError("Recorded arm directory escaped/substituted its group.")
        artifacts = saved.get("artifacts")
        mandatory = {"adapter_summary.json", "declaration.json", "direct_prompts.csv", "initial_state.json",
                     "adapter_history.csv", "adapter_selected_predictions.csv", "adapter_epoch_predictions.csv"}
        epoch_zero = sorted(folder.glob("adapter_epoch_000_*.pt"))
        if (not isinstance(artifacts, dict) or not mandatory <= set(artifacts) or len(epoch_zero) != 1
                or epoch_zero[0].name not in artifacts):
            raise ValueError("The completed arm artifact manifest is incomplete, including its identity checkpoint.")
        for name, digest in artifacts.items():
            if (not isinstance(name, str) or Path(name).name != name or not isinstance(digest, str)
                    or len(digest) != 64 or any(c not in "0123456789abcdef" for c in digest)):
                raise ValueError("Malformed completed artifact identity/digest.")
            path = (folder / name).resolve()
            if path.parent != folder or not path.is_file() or sha256_file(path) != digest:
                raise ValueError("Completed arm artifact changed: " + name)
        summary_path = folder / "adapter_summary.json"
        if str(summary_path) != saved["summary_path"] or sha256_file(summary_path) != saved["summary_sha256"]:
            raise ValueError("Completed summary fingerprint changed.")
        summary = json.loads(summary_path.read_text())
        checkpoint = Path(summary["checkpoint"]).resolve()
        if checkpoint.parent != folder or checkpoint.name not in artifacts:
            raise ValueError("The selected checkpoint must have its own recorded full-byte fingerprint.")
        validate_replication_summary(summary, split)
        if (summary["config"]["training_seed"] != saved["training_seed"]
                or ("control" if summary["config"]["residual_cap"] is None else "bounded") != saved["arm"]):
            raise ValueError("Summary seed/cap is not the declared arm.")
        epochs = verify_saved_arm(folder, summary, split, declared["contract"])
        epoch_zero = sorted(folder.glob("adapter_epoch_000_*.pt"))
        if len(epoch_zero) != 1:
            raise ValueError("One preserved identity checkpoint is required per arm.")
        adapter, _, selection = load_adapter_checkpoint(epoch_zero[0], declared["contract"])
        initial_hash = json_hash({k: tensor_sha256(v) for k, v in adapter.state_dict().items()})
        stamp = json.loads((folder / "initial_state.json").read_text())
        seed = saved["training_seed"]
        if stamp != {"training_seed": seed, "split_seed": 42, "sha256": initial_hash} or selection["epoch"] != 0:
            raise ValueError("Initial checkpoint/seed provenance mismatch.")
        if seed in initial and initial[seed] != initial_hash:
            raise ValueError("Paired arm initial states differ.")
        initial[seed] = initial_hash
        records.append({"training_seed": seed, "arm": saved["arm"], "source": "new_replication",
                        "directory": str(folder), "summary": summary, "epochs": epochs})
    if initial[43] == initial[44]:
        raise ValueError("Distinct training seeds unexpectedly share identical initialization.")
    return records


### 13.2 Compute paired seed differences and preserve every gate

In [ ]:
def replication_statistics(table):
    metrics = ["seen_top1", "unseen_top1", "h1", "h5", "unseen_diseased_accuracy", "unseen_healthy_accuracy", "false_healthy_rate"]
    required = {"training_seed", "arm", *metrics}
    expected = {(seed, arm) for seed in (42, 43, 44) for arm in ("control", "bounded")}
    if (not required <= set(table.columns) or len(table) != 6 or table.duplicated(["training_seed", "arm"]).any()
            or set(zip(table.training_seed, table.arm)) != expected):
        raise ValueError("Statistics require exactly both arms of all three seeds, including seed42.")
    values = table[metrics].to_numpy(dtype=np.float64)
    if not np.isfinite(values).all() or (values < 0).any() or (values > 1).any():
        raise ValueError("Statistics require finite fraction-valued metrics.")
    stats, paired = [], []
    for arm in ("control", "bounded"):
        part = table.loc[table.arm == arm]
        for metric in metrics:
            stats.append({"arm": arm, "metric": metric, "n": len(part), "mean": float(part[metric].mean()),
                          "sample_std": float(part[metric].std(ddof=1))})
    for seed in (42, 43, 44):
        part = table.loc[table.training_seed == seed].set_index("arm")
        paired.append({"training_seed": seed, **{metric + "_delta": float(part.loc["bounded", metric] - part.loc["control", metric]) for metric in metrics}})
    return {"selected_table": table.sort_values(["training_seed", "arm"]).reset_index(drop=True),
            "descriptive_statistics": pd.DataFrame(stats), "paired_differences": pd.DataFrame(paired)}


def summarize_gate_decisions(decisions):
    if len(decisions) != 3 or sorted(x["training_seed"] for x in decisions) != [42, 43, 44]:
        raise ValueError("Report the existing gate for every seed, including42.")
    passing = [x["training_seed"] for x in decisions if x["decision"]["supports_exploratory_success"]]
    return {"all_seeds_pass": len(passing) == 3, "passing_training_seeds": sorted(passing),
            "failed_training_seeds": sorted(set((42, 43, 44)) - set(passing))}


def build_replication_report(records, split):
    if len(records) != 6 or {(x["training_seed"], x["arm"]) for x in records} != {(s, a) for s in (42, 43, 44) for a in ("control", "bounded")}:
        raise ValueError("Report requires all four new arms and both historical references.")
    rows, per_pair, epochs, trajectories, decisions = [], [], [], [], []
    for item in records:
        s, seed = item["summary"], item["training_seed"]
        validate_replication_summary(s, split, historical=seed == 42)
        d, selected = s["diagnostics"]["val_unseen"], s["selected"]
        memory_readings = [x["train_peak_allocated_bytes"] for x in s["history"][1:]
                           if x["train_peak_allocated_bytes"] is not None]
        rows.append({"training_seed": seed, "arm": item["arm"], "source": item["source"],
            "selected_epoch": selected["epoch"], **{k: selected[k] for k in ("seen_top1", "seen_top5", "unseen_top1", "unseen_top5", "h1", "h5")},
            "unseen_diseased_hits": d["diseased_correct"], "unseen_diseased_count": d["diseased_count"],
            "unseen_diseased_accuracy": d["diseased_correct"] / d["diseased_count"],
            "unseen_healthy_accuracy": d["healthy_correct"] / d["healthy_count"],
            "false_healthy": d["false_healthy"], "false_healthy_rate": d["false_healthy"] / d["diseased_count"],
            "fitting_prediction_preference": d["predicted_fitting_composition"] / d["count"],
            "checkpoint": s["checkpoint"], "directory": item["directory"],
            "fitting_seconds": sum(x["train_elapsed_seconds"] for x in s["history"][1:]),
            "peak_fitting_allocated_bytes": max(memory_readings, default=None)})
        for pair in s["diagnostics"]["per_pair"]:
            per_pair.append({"training_seed": seed, "arm": item["arm"], **pair})
        for epoch in item["epochs"]:
            epochs.append({"training_seed": seed, "arm": item["arm"], **epoch})
        for epoch in s["history"]:
            trajectories.append({"training_seed": seed, "arm": item["arm"], **epoch})
    for seed in (42, 43, 44):
        pair = {x["arm"]: x["summary"] for x in records if x["training_seed"] == seed}
        decisions.append({"training_seed": seed, "decision": compare_pilot_summaries(pair["control"], pair["bounded"])})
    result = replication_statistics(pd.DataFrame(rows))
    result.update({"gate_decisions": decisions, "gate_summary": summarize_gate_decisions(decisions),
                   "per_pair": pd.DataFrame(per_pair), "epoch_subgroups": pd.DataFrame(epochs), "trajectories": pd.DataFrame(trajectories),
                   "interpretation": "Descriptive variability across three training seeds on one fixed split. No formal significance or final-test claim; every disease-gate failure is retained."})
    return result


### 13.3 Save and plot complete recorded comparisons

In [ ]:
def load_and_report_replications(directory, split, table, config):
    records = load_historical_reference(split, table, config) + read_completed_replication(directory, split, table, config)
    report = build_replication_report(records, split)
    directory = require_external_path(directory)
    names = {"selected_table": "replication_selected.csv", "descriptive_statistics": "replication_statistics.csv",
             "paired_differences": "replication_paired_differences.csv", "per_pair": "replication_per_pair.csv",
             "epoch_subgroups": "replication_epoch_subgroups.csv", "trajectories": "replication_trajectories.csv"}
    targets = [directory / name for name in names.values()] + [directory / "replication_interpretation.json"]
    if any(path.exists() for path in targets):
        if not all(path.is_file() for path in targets):
            raise ValueError("Partial prior report exists; preserve it and diagnose before rerunning.")
        expected_interpretation = {"gate_decisions": report["gate_decisions"],
            "gate_summary": report["gate_summary"], "interpretation": report["interpretation"]}
        if json.loads((directory / "replication_interpretation.json").read_text()) != expected_interpretation:
            raise ValueError("Recorded interpretation/gate decisions changed.")
        # Reopening a report is read-only; never rewrite prior tables or silently substitute new results.
        for key, name in names.items():
            existing = pd.read_csv(directory / name, float_precision="round_trip")
            # Compare the same CSV representation: unavailable CPU memory becomes NaN on read.
            from io import StringIO
            expected = pd.read_csv(StringIO(report[key].to_csv(index=False)), float_precision="round_trip")
            if not existing.equals(expected):
                raise ValueError("Existing report differs; preserve it and investigate: " + name)
    else:
        for key, name in names.items():
            report[key].to_csv(directory / name, index=False, mode="x")
        write_exclusive_json(directory / "replication_interpretation.json", {"gate_decisions": report["gate_decisions"],
            "gate_summary": report["gate_summary"], "interpretation": report["interpretation"]})
    return report


def plot_replication_report(report):
    """Only recorded complete results are plotted; no unexecuted placeholders."""
    selected, history = report["selected_table"], report["trajectories"]
    colors = {"control": "#3b6f9c", "bounded": "#b65728"}
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    for arm in ("control", "bounded"):
        part = selected.loc[selected.arm == arm].sort_values("training_seed")
        for ax, metric, title in zip(axes, ("h1", "unseen_diseased_accuracy"),
                                    ("Selected development H@1", "Selected unseen-diseased composition accuracy (N=1496)")):
            ax.plot(part.training_seed, part[metric] * 100, marker="o", label=arm, color=colors[arm],
                    linestyle="-" if arm == "bounded" else "--")
            ax.set(title=title, xlabel="Training seed; split seed fixed at42", ylabel="Percent")
            ax.set_xticks([42, 43, 44]); ax.grid(alpha=.2); ax.legend()
    plt.show()
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    for (seed, arm), part in history.groupby(["training_seed", "arm"]):
        axes[0].plot(part.epoch, part.h1 * 100, marker=".", label=str(seed) + "/" + arm)
        axes[1].plot(part.epoch, part.val_unseen_disease_loss, marker=".", label=str(seed) + "/" + arm)
    axes[0].set(title="All recorded epochs: development H@1", xlabel="Epoch", ylabel="Percent")
    axes[1].set(title="Internally unseen validation disease loss", xlabel="Epoch", ylabel="Cross-entropy")
    for ax in axes:
        ax.grid(alpha=.2); ax.legend(fontsize=8)
    plt.show()


## 14. Safe metadata inspection

This cell reads only training metadata and prints the frozen identities. It creates no run, cache, checkpoint or model.

In [ ]:
validate_replication_config(CONFIG)
SPLIT = build_development_split(ROOT / "DPD_Small_train.csv", seed=CONFIG["split_seed"])
verify_split_identity(SPLIT)
DIRECT_TABLE = build_prompt_table(SPLIT)
print({"split_seed": CONFIG["split_seed"], "training_seeds": CONFIG["training_seeds"],
       "fold_counts": {role: len(getattr(SPLIT, role)) for role in ("fit", "val_seen", "val_unseen")},
       "split": verify_split_identity(SPLIT), "cache_sha256": CONFIG["feature_cache_sha256"]})
display(DIRECT_TABLE.head(8))


## 15. Explicit owner execution

Fresh kernel: enable `LOAD_FEATURE_CACHE`, `RUN_REPLICATIONS` and `BUILD_REPLICATION_REPORT`, then Run All. The first stage verifies the existing cache; it never rebuilds it. Exactly four five-epoch arms follow. Default controls below skip all artifact/model/training activity.

In [ ]:
if LOAD_FEATURE_CACHE:
    FEATURE_CACHE = load_feature_cache(CONFIG["feature_cache_path"], CONFIG["feature_cache_sha256"], SPLIT, DIRECT_TABLE, CONFIG)
    verify_cached_frozen_reference(FEATURE_CACHE, SPLIT)
    print({"cache_verified": True, "image_encoding": False, "fold_shapes": {
        role: list(FEATURE_CACHE["folds"][role]["features"].shape) for role in ("fit", "val_seen", "val_unseen")}})
else:
    print("LOAD_FEATURE_CACHE=False: no cache or image files are loaded.")


In [ ]:
if RUN_REPLICATIONS:
    if FEATURE_CACHE is None:
        raise RuntimeError("Explicitly load/verify the cache first.")
    REPLICATION_RESULTS = run_replications(SPLIT, DIRECT_TABLE, CONFIG, FEATURE_CACHE)
else:
    print("RUN_REPLICATIONS=False: no adapter training or development evaluation.")


In [ ]:
if BUILD_REPLICATION_REPORT:
    directory = REPLICATION_RESULTS["group_directory"] if REPLICATION_RESULTS is not None else REPORT_GROUP_DIRECTORY
    if directory is None:
        raise RuntimeError("Provide a completed replication group; partial results are never treated as complete.")
    REPLICATION_REPORT = load_and_report_replications(directory, SPLIT, DIRECT_TABLE, CONFIG)
    display(REPLICATION_REPORT["selected_table"])
    display(REPLICATION_REPORT["descriptive_statistics"])
    display(REPLICATION_REPORT["paired_differences"])
    print(REPLICATION_REPORT["interpretation"])
    plot_replication_report(REPLICATION_REPORT)
else:
    print("BUILD_REPLICATION_REPORT=False: no recorded-run reads or plots. No new results are claimed.")


## 16. Interpretation and owner handoff

Keep seed42 reference files and all selected epochs unchanged. The primary scorer remains direct323-candidate float64 cosine similarity, with original global IDs and ascending-ID exact ties. Final333-candidate evaluation is a separate future protocol; no official-test images or labels are used here.

Evaluate the H@1 gain together with unseen-diseased hits, healthy accuracy, false-healthy errors, per-pair support and H@5. Preserve every failed gate in the report; never choose an alternate epoch after examining subgroups. Mean/SD are descriptive across training randomness only. A falling training loss or one favorable seed is insufficient evidence of better disease transfer.

No replication has been run in this delivery. If an owner run is interrupted, preserve its unique directory and failure record, diagnose the cause, and start a fresh explicit attempt. No checkpoint resumption or automatic retry is implemented. Do not Run All the historical M4 notebook again.